In [ ]:
!apt-get update -q
!apt-get install -y postgresql postgresql-contrib libpq-dev python3-dev build-essential -q

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lis

In [ ]:
!service postgresql start


 * Starting PostgreSQL 14 database server
   ...done.


In [ ]:
# ── Mount Google Drive & persist PostgreSQL data ──────────────
import os, subprocess, shutil
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DB_DIR = '/content/drive/MyDrive/postgres_milestone_db'
PG_DATA_DIR  = '/var/lib/postgresql/14/main'
BACKUP_PATH  = '/content/drive/MyDrive/postgres_milestone_db/milestone1_users.sql'

os.makedirs(DRIVE_DB_DIR, exist_ok=True)

# Start PostgreSQL first
subprocess.run(['service', 'postgresql', 'start'], capture_output=True)

# Set password
subprocess.run(['sudo','-u','postgres','psql','-c',
                "ALTER USER postgres PASSWORD 'postgres';"], capture_output=True)

# Create DB if not exists
result = subprocess.run(['sudo','-u','postgres','psql','-lqt'], capture_output=True, text=True)
if 'milestone1_users' not in result.stdout:
    subprocess.run(['sudo','-u','postgres','psql','-c','CREATE DATABASE milestone1_users;'], capture_output=True)
    print('✅ Database created fresh')

# ── Restore from Drive backup if it exists ────────────────────
if os.path.exists(BACKUP_PATH):

    print('📂 Found Drive backup — cleaning old tables...')

    subprocess.run(
        ['sudo','-u','postgres','psql','-d','milestone1_users','-c',
         'DROP SCHEMA public CASCADE; CREATE SCHEMA public;'],
        capture_output=True
    )

    print('♻️ Restoring database from backup...')

    subprocess.run(
        f'sudo -u postgres psql milestone1_users < "{BACKUP_PATH}"',
        shell=True, capture_output=True
    )

    print('✅ Database restored from Google Drive!')
    print('✅ Database restored from Google Drive!')
else:
    print('ℹ️ No backup found — starting fresh (will save on next run)')

print('\n✅ PostgreSQL ready. Drive connected.')
print(f'📁 Backups saved to: {DRIVE_DB_DIR}')


Mounted at /content/drive
✅ Database created fresh
📂 Found Drive backup — cleaning old tables...
♻️ Restoring database from backup...
✅ Database restored from Google Drive!
✅ Database restored from Google Drive!

✅ PostgreSQL ready. Drive connected.
📁 Backups saved to: /content/drive/MyDrive/postgres_milestone_db


In [ ]:
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"


ALTER ROLE


In [ ]:
import subprocess
result = subprocess.run(['sudo', '-u', 'postgres', 'psql', '-lqt'], capture_output=True, text=True)
if 'milestone1_users' not in result.stdout:
    subprocess.run(['sudo', '-u', 'postgres', 'psql', '-c', 'CREATE DATABASE milestone1_users;'])
    print('Database created.')
else:
    print('Database already exists.')

Database already exists.


In [ ]:
import psycopg2
conn = psycopg2.connect(dbname='milestone1_users', user='postgres', password='postgres', host='localhost')
print('✅ PostgreSQL Connected Successfully!')
conn.close()

✅ PostgreSQL Connected Successfully!


In [ ]:
!pip install streamlit psycopg2-binary bcrypt pyjwt watchdog pyngrok -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 103.5 MB/s eta 0:00:00


In [ ]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok nltk streamlit-option-menu plotly textstat PyPDF2 transformers torch sentencepiece accelerate pandas "bitsandbytes>=0.40.0" psycopg2-binary -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 42.1 MB/s eta 0:00:00


In [ ]:
%%writefile text_readability.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text = text
        self.num_sentences = textstat.sentence_count(text)
        self.num_words = textstat.lexicon_count(text, removepunct=True)
        self.num_syllables = textstat.syllable_count(text)
        self.complex_words = textstat.difficult_words(text)
        self.char_count = textstat.char_count(text)

    def get_all_metrics(self):
        return {
            "Flesch Reading Ease": textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "SMOG Index": textstat.smog_index(self.text),
            "Gunning Fog": textstat.gunning_fog(self.text),
            "Coleman-Liau": textstat.coleman_liau_index(self.text)
        }


Writing text_readability.py


In [ ]:
%%writefile engine.py
import os
import torch
import streamlit as st
import nltk
from nltk.tokenize import sent_tokenize

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

TRANSFORMERS_AVAILABLE = False
BNB_AVAILABLE = False
try:
    from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
    TRANSFORMERS_AVAILABLE = True
    try:
        from transformers import BitsAndBytesConfig
        BNB_AVAILABLE = True
    except ImportError:
        pass
except ImportError:
    TRANSFORMERS_AVAILABLE = False

LANG_CODES = {
    "English": "eng_Latn", "Hindi": "hin_Deva", "Tamil": "tam_Taml",
    "Kannada": "kan_Knda", "Telugu": "tel_Telu", "Marathi": "mar_Deva",
    "Bengali": "ben_Beng"
}

@st.cache_resource(show_spinner=False)
def load_translation_model():
    """Load NLLB multilingual translation model."""
    if not TRANSFORMERS_AVAILABLE:
        return None
    try:
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
        model = AutoModelForSeq2SeqLM.from_pretrained(
            "facebook/nllb-200-distilled-600M",
            device_map = "auto" if torch.cuda.is_available() else None
        )
        return {"tokenizer": tokenizer, "model": model}
    except Exception as e:
        print(f"Translation model load failed: {e}")
        return None

def translate_text(text, target_lang, translation_model):
    """Translate text to target language using NLLB model."""
    if target_lang == "English" or not text.strip():
        return text
    lang_code = LANG_CODES.get(target_lang)
    if not lang_code:
        return text
    if translation_model is None:
        return text + f"\n\n[⚠️ Translation model unavailable. Install: pip install transformers]"
    try:
        tokenizer = translation_model["tokenizer"]
        model     = translation_model["model"]
        device    = next(model.parameters()).device
        # Split into chunks to handle long text
        sentences = text.split(". ")
        chunks, curr = [], []
        curr_len = 0
        for s in sentences:
            wlen = len(s.split())
            if curr_len + wlen > 200 and curr:
                chunks.append(". ".join(curr) + ".")
                curr, curr_len = [s], wlen
            else:
                curr.append(s)
                curr_len += wlen
        if curr:
            chunks.append(". ".join(curr))
        translated_parts = []
        for chunk in chunks:
            if not chunk.strip():
                continue
            inputs = tokenizer(chunk, return_tensors="pt",
                               padding=True, truncation=True,
                               max_length=512).to(device)
            target_id = target_id = tokenizer.convert_tokens_to_ids(lang_code)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    forced_bos_token_id=target_id,
                    max_new_tokens=512,
                    num_beams=4,
                    early_stopping=True
                )
            translated_parts.append(
                tokenizer.decode(outputs[0], skip_special_tokens=True)
            )
        return " ".join(translated_parts)
    except Exception as e:
        return text + f"\n\n[Translation error: {str(e)}]"


@st.cache_resource(show_spinner=False)
def load_summarization_models(quantization_level="4-bit"):
    models = {}
    if not TRANSFORMERS_AVAILABLE:
        return models
    kwargs = {"device_map": "auto"}
    if BNB_AVAILABLE:
        if quantization_level == "8-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        elif quantization_level == "4-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    try:
        models['bart'] = {
            'tokenizer': AutoTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6"),
            'model': AutoModelForSeq2SeqLM.from_pretrained("sshleifer/distilbart-cnn-12-6", **kwargs)
        }
    except Exception as e:
        print(f"BART load failed: {e}"); models['bart'] = None
    try:
        models['pegasus'] = {
            'tokenizer': AutoTokenizer.from_pretrained("google/pegasus-cnn_dailymail"),
            'model': AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-cnn_dailymail", **kwargs)
        }
    except Exception as e:
        print(f"Pegasus load failed: {e}"); models['pegasus'] = None
    try:
        for t5m in ["google/flan-t5-base", "google/flan-t5-small"]:
            try:
                models['flan-t5'] = {
                    'tokenizer': AutoTokenizer.from_pretrained(t5m),
                    'model': AutoModelForSeq2SeqLM.from_pretrained(t5m, **kwargs)
                }
                break
            except Exception:
                continue
        if 'flan-t5' not in models:
            models['flan-t5'] = None
    except Exception as e:
        print(f"FLAN-T5 load failed: {e}"); models['flan-t5'] = None
    return models

@st.cache_resource(show_spinner=False)
def load_paraphrase_models(quantization_level="4-bit"):
    models = {}
    if not TRANSFORMERS_AVAILABLE:
        return models
    kwargs = {"device_map": "auto"}
    if BNB_AVAILABLE:
        if quantization_level == "8-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        elif quantization_level == "4-bit":
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    try:
        try:
            models['flan_t5'] = {
                'tokenizer': AutoTokenizer.from_pretrained("Vamsi/T5_Paraphrase_Paws"),
                'model': AutoModelForSeq2SeqLM.from_pretrained("Vamsi/T5_Paraphrase_Paws", **kwargs)
            }
        except Exception:
            models['flan_t5'] = {
                'tokenizer': AutoTokenizer.from_pretrained("google/flan-t5-small"),
                'model': AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small", **kwargs)
            }
        try:
            models['bart'] = {
                'tokenizer': AutoTokenizer.from_pretrained("eugenesiow/bart-paraphrase"),
                'model': AutoModelForSeq2SeqLM.from_pretrained("eugenesiow/bart-paraphrase", **kwargs)
            }
        except Exception:
            models['bart'] = None
    except Exception:
        pass
    return models

def simple_text_summarization(text, summary_length):
    try:
        sentences = sent_tokenize(text)
        if len(sentences) <= 2:
            return text[:100] + "..." if len(text) > 100 else text
        if summary_length == "Short":
            return " ".join(sentences[:max(1, len(sentences) // 4)])
        elif summary_length == "Medium":
            return " ".join(sentences[:max(2, len(sentences) // 2)])
        else:
            return " ".join(sentences[:max(3, int(len(sentences) * 0.75))])
    except:
        return text[:150] + "..." if len(text) > 150 else text

def _detect_hallucination(original_text, generated_text):
    from collections import Counter
    gen_words = generated_text.split()
    orig_words = set(original_text.lower().split())
    if len(gen_words) < 3:
        return True
    word_counts = Counter(w.lower().strip(".,!?();:'\"") for w in gen_words)
    most_common_count = word_counts.most_common(1)[0][1] if word_counts else 0
    if most_common_count > len(gen_words) * 0.5 and len(gen_words) > 20:
        return True
    gen_clean = [w.lower().strip(".,!?();:'\"") for w in gen_words]
    novel_words = [w for w in gen_clean if w not in orig_words and len(w) > 3]
    if len(novel_words) > len(gen_words) * 0.85 and len(gen_words) > 30:
        return True
    return False

def local_summarize(text, summary_length, model_type, models_dict, target_lang="English"):
    model_key = model_type.lower()
    if model_key not in models_dict or models_dict[model_key] is None:
        st.warning(f"⚠️ {model_type} not available. Using fallback.")
        return simple_text_summarization(text, summary_length)
    model_info = models_dict[model_key]
    tokenizer  = model_info['tokenizer']
    model      = model_info['model']
    input_length = len(tokenizer.encode(text))
    input_word_count = len(text.split())
    is_long = input_word_count >= 1000
    if is_long:
        length_config = {
            "Short":  {"max_length": min(300, max(150, input_length//4)),   "min_length": min(100, max(50, input_length//6))},
            "Medium": {"max_length": min(700, max(400, int(input_length*.5))), "min_length": min(350, max(200, input_length//3))},
            "Long":   {"max_length": min(1500, max(800, int(input_length*.85))), "min_length": min(700, max(500, int(input_length*.6)))}
        }
    else:
        safe_max = max(60, int(input_length * 0.95))
        length_config = {
            "Short":  {"max_length": min(60,  max(20, input_length//4)),  "min_length": min(10, max(5, input_length//6))},
            "Medium": {"max_length": min(150, max(40, input_length//2)),  "min_length": min(25, max(12, input_length//4))},
            "Long":   {"max_length": min(safe_max, max(80, int(input_length*.9))), "min_length": min(50, max(25, input_length//2))}
        }
    config = length_config.get(summary_length, length_config["Medium"])
    config["min_length"] = min(config["min_length"], config["max_length"] - 5)
    config["min_length"] = max(config["min_length"], 5)
    if model_key == 'flan-t5':
        prompts = {
            "Short":  f"Write a brief 2-3 sentence summary: {text}",
            "Medium": f"Write a detailed summary covering main points: {text}",
            "Long":   f"Write a comprehensive summary covering all key points: {text}"
        }
        prompt = prompts.get(summary_length, prompts["Medium"])
    else:
        prompt = text
    try:
        with st.spinner(f"🧠 {model_type} generating summary..."):
            device = next(model.parameters()).device
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                               max_length=1024, padding=True).to(device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=config["max_length"],
                    min_new_tokens=config["min_length"],
                    num_beams=2,
                    length_penalty={"Short": 0.6, "Medium": 1.0, "Long": 1.8}.get(summary_length, 1.0),
                    no_repeat_ngram_size=3,
                    early_stopping=True,
                    use_cache=True,
                    repetition_penalty=1.5,
                )
            summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
            if _detect_hallucination(text, summary) or not summary.strip():
                summary = simple_text_summarization(text, summary_length)
            return summary
    except Exception as e:
        st.error(f"❌ {model_type} error: {str(e)}")
        return simple_text_summarization(text, summary_length)

def apply_fallback_paraphrasing(text, complexity):
    subs = {
        "Simple":   {"utilize":"use","facilitate":"help","fundamental":"basic","however":"but","moreover":"also"},
        "Neutral":  {"use":"utilize","help":"assist","basic":"fundamental","but":"however","also":"furthermore"},
        "Advanced": {"use":"leverage","help":"facilitate","basic":"foundational","but":"nevertheless","also":"moreover","show":"demonstrate","important":"paramount"},
    }
    sub_dict = subs.get(complexity, subs["Neutral"])
    result = []
    for word in text.split():
        clean = word.strip(".,!?();:'\"").lower()
        if clean in sub_dict:
            new = sub_dict[clean]
            if word[0].isupper(): new = new.capitalize()
            result.append(new)
        else:
            result.append(word)
    return " ".join(result)

def paraphrase_with_model(text, complexity, style, model_type, models_dict, target_lang="English"):
    model_key = model_type.lower().replace('-', '_')
    try:
        model_info = models_dict.get(model_key)
        if model_info is None:
            return apply_fallback_paraphrasing(text, complexity)
        tokenizer = model_info['tokenizer']
        model     = model_info['model']
        device    = next(model.parameters()).device
        sentences = sent_tokenize(text)
        chunks, curr, curr_len = [], [], 0
        for s in sentences:
            slen = len(s.split())
            if curr_len + slen > 80 and curr:
                chunks.append(" ".join(curr)); curr = [s]; curr_len = slen
            else:
                curr.append(s); curr_len += slen
        if curr: chunks.append(" ".join(curr))
        results = []
        for chunk in chunks:
            token_count = len(tokenizer.encode(chunk))
            if model_key == 'flan_t5':
                prompt = f"paraphrase the following text using different words and sentence structure: {chunk} </s>"
            else:
                prompt = f"paraphrase: {chunk}"
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                               max_length=512, padding="max_length").to(device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max(150, int(token_count * 1.5)),
                    min_new_tokens=max(10,  int(token_count * 0.6)),
                    num_beams=1,
                    no_repeat_ngram_size=3,
                    repetition_penalty=1.8,
                    use_cache=True
                )
            paraphrased = tokenizer.decode(outputs[0], skip_special_tokens=True)
            results.append(paraphrased if len(paraphrased.strip()) > 10 else chunk)
        return " ".join(results) or apply_fallback_paraphrasing(text, complexity)
    except Exception as e:
        st.error(f"❌ Paraphrasing error ({model_type}): {str(e)}")
        return apply_fallback_paraphrasing(text, complexity)


Writing engine.py


In [ ]:
!pip install plotly PyPDF2 textstat py-readability-metrics

In [ ]:
%%writefile app.py
import streamlit as st
import psycopg2
import jwt
import datetime
import bcrypt
import sqlite3
import os
import re
import time
import smtplib
import secrets
import hmac
import hashlib
import struct
import random
import base64
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from streamlit_option_menu import option_menu
import plotly.graph_objects as go
from collections import Counter
from wordcloud import WordCloud
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import PyPDF2
import textstat
import pandas as pd
from text_readability import ReadabilityAnalyzer
import engine

EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
SECRET_KEY     = os.getenv("JWT_secret_key", "dev_secret_key")
ADMIN_EMAIL_ID = os.getenv("ADMIN_EMAIL_ID")
ADMIN_PASSWORD = os.getenv("ADMIN_PASSWORD")
EMAIL_ADDRESS  = os.getenv("EMAIL_ADDRESS")
ALGORITHM      = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30
OTP_EXPIRY_MINUTES = 10
MAX_LOGIN_ATTEMPTS = 3
LOCKOUT_SECONDS    = 300

DB_NAME     = os.getenv("DB_NAME",     "milestone1_users")
DB_USER     = os.getenv("DB_USER",     "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "postgres")
DB_HOST     = os.getenv("DB_HOST",     "localhost")
DB_PORT     = os.getenv("DB_PORT",     "5432")
# ── Forgot Password Session State ──
if "forgot_stage" not in st.session_state:
    st.session_state["forgot_stage"] = "email"

if "reset_email" not in st.session_state:
    st.session_state["reset_email"] = None

if "otp_token" not in st.session_state:
    st.session_state["otp_token"] = None

if "otp_stage" not in st.session_state:
    st.session_state["otp_stage"] = False

if "otp_email" not in st.session_state:
    st.session_state["otp_email"] = None

SUPPORTED_LANGUAGES = ["English", "Hindi", "Tamil", "Kannada", "Telugu", "Marathi", "Bengali"]

SECURITY_QUESTIONS = [
    "What is your mother's maiden name?",
    "What was the name of your first pet?",
    "What is your favorite teacher's name?",
    "What is your birthplace?",
    "What is your favorite food?",
    "What is your childhood nickname?"
]
if 'summarization_models' not in st.session_state:
    with st.spinner("Loading AI models..."):
        st.session_state.summarization_models = engine.load_summarization_models()
        st.session_state.paraphrase_models    = engine.load_paraphrase_models()
        st.session_state.translation_model    = engine.load_translation_model()

st.set_page_config(page_title="TextMorph Advanced Text Summarization and Paraphrasing", layout="wide",
                   initial_sidebar_state="expanded")

def apply_clean_theme():
    st.markdown("""
<style>

@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');

/* ================================= */
/* REMOVE STREAMLIT DEFAULT ELEMENTS */
/* ================================= */

#MainMenu {visibility: hidden;}
footer {visibility: hidden;}

/* keep header so sidebar toggle works */
header {
    background: transparent !important;
}


/* ================================= */
/* GRADIENT BACKGROUND (FROM UI 1) */
/* ================================= */

.stApp {
    background: linear-gradient(135deg,#ff6ec7,#845ec2,#2c73d2,#00c9a7,#f9f871);
    background-size: 400% 400%;
    animation: gradientMove 12s ease infinite;
    font-family: 'Inter', sans-serif;
    color: white;
}

/* ================================= */
/* GLASS MAIN CONTAINERS */
/* ================================= */

.block-container{
    background: rgba(255,255,255,0.10);
    backdrop-filter: blur(25px);
    border-radius: 20px;
    padding: 30px;
}

@keyframes gradientMove {
    0%{background-position:0% 50%;}
    50%{background-position:100% 50%;}
    100%{background-position:0% 50%;}
}

.gradient-text{
background:linear-gradient(135deg,#00c9a7,#845ec2);
-webkit-background-clip:text;
color:transparent;
font-weight:700;
}

/* ================================= */
/* SIDEBAR GLASS PANEL */
/* ================================= */

section[data-testid="stSidebar"]{

    background: rgba(255,255,255,0.08);

    backdrop-filter: blur(30px);

    border-right: 1px solid rgba(255,255,255,0.25);

}

section[data-testid="stSidebar"] > div{
    background: transparent !important;
}

section[data-testid="stSidebar"] *{
    color:white !important;
}


/* ================================= */
/* GLASS INPUT STYLE */
/* ================================= */

.stTextInput > div > div{
    background: rgba(255,255,255,0.12) !important;
    backdrop-filter: blur(18px);

    border-radius: 50px;

    border: 1px solid rgba(255,255,255,0.35);

    transition: all 0.2s ease;
}

/* remove internal dark background */

div[data-baseweb="base-input"]{
    background: transparent !important;
}

/* actual input */

.stTextInput input{
    background: transparent !important;

    color: white !important;

    border: none !important;

    padding: 14px 18px;

    font-size: 15px;
}

/* placeholder */

.stTextInput input::placeholder{
    color: rgba(255,255,255,0.65);
}

/* focus effect */

.stTextInput input:focus{

    outline: none !important;

}

/* glow when active */

.stTextInput > div > div:focus-within{

    border: 1px solid rgba(255,255,255,0.65);

    box-shadow: 0 0 10px rgba(255,255,255,0.25);

}

/* ================================= */
/* REMOVE STREAMLIT INPUT BACKGROUND */
/* ================================= */

div[data-baseweb="input"]{
    background: transparent !important;
    border: none !important;
}

div[data-baseweb="base-input"]{
    background: transparent !important;
}

/* remove wrapper color */

.stTextInput > div{
    background: transparent !important;
}

/* ensure glass layer stays visible */

.stTextInput > div > div{
    background: rgba(255,255,255,0.15) !important;
}

/* ================================= */
/* PASSWORD ICON FIX */
/* ================================= */

.stTextInput div[data-baseweb="input"] > div{
    background: transparent !important;
}

/* ================================= */
/* TEXTAREA GLASS STYLE */
/* ================================= */

.stTextArea > div{
    background: transparent !important;
}

.stTextArea > div > div{
    background: rgba(255,255,255,0.15) !important;
    backdrop-filter: blur(20px);
    border-radius: 14px;
    border: 1px solid rgba(255,255,255,0.35);
}

/* textarea */

.stTextArea textarea{
    background: transparent !important;
    color: white !important;
    border: none !important;
    outline: none !important;
    padding: 16px;
}

/* remove baseweb background */

div[data-baseweb="textarea"]{
    background: transparent !important;
}

/* remove black focus border */

div[data-baseweb="textarea"]:focus-within{
    border: 1px solid rgba(255,255,255,0.35) !important;
    box-shadow: none !important;
}

/* browser focus */

textarea:focus{
    outline: none !important;
    box-shadow: none !important;
}
/* ================================= */
/* BUTTON STYLE */
/* ================================= */

.stButton > button,
.stFormSubmitButton > button{

    background: linear-gradient(135deg,#00c9a7,#845ec2) !important;
    color:white !important;

    border-radius:12px !important;
    border:none !important;

    padding:10px 20px !important;
    font-weight:600;
}

.stButton > button:hover,
.stFormSubmitButton > button:hover{

    background: linear-gradient(135deg,#2c73d2,#ff6ec7) !important;

    transform: translateY(-2px);

    box-shadow:0 4px 12px rgba(0,0,0,0.3);
}


/* ================================= */
/* NAV TABS */
/* ================================= */

.stTabs [data-baseweb="tab-list"]{

    background: rgba(255,255,255,0.12);

    backdrop-filter: blur(20px);

    border-radius: 14px;

}

.stTabs [data-baseweb="tab"]{

    color: white;

    font-weight: 500;

    border-radius: 10px;

}

/* ACTIVE TAB */

.stTabs [aria-selected="true"]{

    background: linear-gradient(135deg,#00c9a7,#845ec2);

    color: white !important;

    box-shadow: 0 4px 12px rgba(0,0,0,0.3);

}

/* ================================= */
/* SELECTBOX GLASS STYLE */
/* ================================= */

.stSelectbox div[data-baseweb="select"]{

    background: rgba(255,255,255,0.12) !important;

    backdrop-filter: blur(20px);

    border-radius: 12px;

    border: 1px solid rgba(255,255,255,0.25);

}

div[data-baseweb="select"]{
    background: rgba(255,255,255,0.12) !important;
}

/* ================================= */
/* METRIC CARDS */
/* ================================= */

[data-testid="metric-container"]{

    background: rgba(255,255,255,0.12);

    backdrop-filter: blur(20px);

    border-radius: 16px;

    border: 1px solid rgba(255,255,255,0.25);

}


/* ================================= */
/* PAGE HEADER */
/* ================================= */

.page-header{

    background: rgba(255,255,255,0.12);

    backdrop-filter: blur(20px);

    border-radius:16px;

    padding:24px;

    margin-bottom:20px;

}


/* ================================= */
/* HISTORY CARDS */
/* ================================= */

.hist-card{

    background: rgba(255,255,255,0.10);

    backdrop-filter: blur(20px);

    border-radius:12px;

    padding:16px;

    margin-bottom:10px;

}


/* ================================= */
/* RESULT OUTPUT */
/* ================================= */

.result-box-output{

    background: rgba(255,255,255,0.15);

    backdrop-filter: blur(20px);

    border-left: 4px solid #00c9a7;

    border-radius: 14px;

    padding: 18px;

    color: white;

}


/* ================================= */
/* FLOATING CHAT INPUT */
/* ================================= */

.stChatInputContainer{

    position:fixed;

    bottom:30px;

    left:50%;

    transform:translateX(-50%);

    width:60%;

    background: rgba(255,255,255,0.1);

    backdrop-filter: blur(20px);

    border-radius:25px;

    padding:8px;

}


/* ================================= */
/* SCROLLBAR */
/* ================================= */

::-webkit-scrollbar{
    width:6px;
}

::-webkit-scrollbar-thumb{
    background:white;
    border-radius:10px;
}

/* ================================= */
/* FILE UPLOADER – STYLE 1 */
/* ================================= */

[data-testid="stFileUploader"]{

    background: rgba(255,255,255,0.12) !important;

    backdrop-filter: blur(25px);

    border-radius: 18px;

    border: 1px solid rgba(255,255,255,0.35);

    padding: 12px;

}

/* inner container */

[data-testid="stFileUploader"] section{

    background: transparent !important;

}

/* browse button */

[data-testid="stFileUploader"] button{

    background: linear-gradient(135deg,#00c9a7,#845ec2) !important;

    color:white !important;

    border-radius:12px !important;

    border:none !important;

}

/* ================================= */
/* DATA TABLE GLASS */
/* ================================= */

.stDataFrame{

    background: rgba(255,255,255,0.08);

    backdrop-filter: blur(20px);

    border-radius: 14px;

}

.stDataFrame thead{

    background: rgba(255,255,255,0.15);

}

textarea, input{
    color:white !important;
}

/* ================================= */
/* REMOVE STREAMLIT DARK THEMES */
/* ================================= */

div[data-baseweb="input"],
div[data-baseweb="textarea"],
div[data-baseweb="select"]{

    background: transparent !important;

}

/* remove internal dark containers */

.stTextArea textarea,
.stSelectbox div[data-baseweb="select"],
textarea{

    background: transparent !important;

}

/* ================================= */
/* FORCE ALL TEXT TO WHITE */
/* ================================= */

html, body, p, span, div, label {
    color: white !important;
}

/* headings */

h1,h2,h3,h4,h5,h6{
    color:white !important;
}

/* captions */

.stCaption{
    color:rgba(255,255,255,0.7) !important;
}

/* metric text */

[data-testid="metric-container"] *{
    color:white !important;
}

/* dataframe text */

.stDataFrame div{
    color:white !important;
}

/* table header */

.stDataFrame thead{
    color:white !important;
}

/* expanders */

details summary{
    color:white !important;
}

/* ================================= */
/* DATAFRAME IMPROVED VISIBILITY */
/* ================================= */

.stDataFrame{
    color:white !important;
}

.stDataFrame th{
    color:white !important;
    font-weight:600;
}

.stDataFrame td{
    color:white !important;
}

/* ================================= */
/* FIX SIDEBAR VISIBILITY */
/* ================================= */

section[data-testid="stSidebar"]{
    display:block !important;
    visibility:visible !important;
    width:260px !important;
    min-width:260px !important;
}

section[data-testid="stSidebar"] > div{
    display:block !important;
}

/* keep sidebar above glass container */

.css-1d391kg{
    z-index:1000 !important;
}

/* ===== FIX DATAFRAME DARK BACKGROUND ===== */

[data-testid="stDataFrame"] {
    background: rgba(255,255,255,0.10) !important;
    backdrop-filter: blur(20px);
    border-radius: 12px;
    border: 1px solid rgba(255,255,255,0.25);
}

/* table body */

[data-testid="stDataFrame"] div[role="grid"]{
    background: rgba(255,255,255,0.08) !important;
}

/* table rows */

[data-testid="stDataFrame"] div[role="row"]{
    background: rgba(255,255,255,0.05) !important;
}

/* table header */

[data-testid="stDataFrame"] div[role="columnheader"]{
    background: rgba(255,255,255,0.15) !important;
    color:white !important;
}

/* table cells */

[data-testid="stDataFrame"] div[role="gridcell"]{
    background: transparent !important;
    color:white !important;
}

</style>
""", unsafe_allow_html=True)

apply_clean_theme()

# ── DB connection ──────────────────────────────────────────
def get_connection():
    return psycopg2.connect(dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD,
                            host=DB_HOST, port=DB_PORT)


def create_tables():
    conn = get_connection()
    cur = conn.cursor()

    # USERS TABLE FIRST
    cur.execute("""
    CREATE TABLE IF NOT EXISTS users (
      id SERIAL PRIMARY KEY,
      username VARCHAR(100) UNIQUE NOT NULL,
      email VARCHAR(150) UNIQUE NOT NULL,
      password TEXT NOT NULL,
      role VARCHAR(20) DEFAULT 'user',
      is_locked BOOLEAN DEFAULT FALSE,
      security_question TEXT NOT NULL,
      security_answer TEXT NOT NULL,
      avatar TEXT,
      created_at TIMESTAMP DEFAULT NOW()
    );
    """)

    cur.execute("ALTER TABLE users ADD COLUMN IF NOT EXISTS avatar TEXT")

    # PASSWORD HISTORY
    cur.execute("""
    CREATE TABLE IF NOT EXISTS password_history (
        id SERIAL PRIMARY KEY,
        email VARCHAR(150) NOT NULL,
        password TEXT NOT NULL,
        set_at TIMESTAMP DEFAULT NOW(),
        FOREIGN KEY (email) REFERENCES users(email)
        ON DELETE CASCADE
        ON UPDATE CASCADE
    );
    """)

    # LOGIN ATTEMPTS
    cur.execute("""
    CREATE TABLE IF NOT EXISTS login_attempts (
        email VARCHAR(150) PRIMARY KEY,
        attempts INTEGER DEFAULT 0,
        last_attempt DOUBLE PRECISION DEFAULT 0
    );
    """)

    # USER HISTORY
    cur.execute("""
    CREATE TABLE IF NOT EXISTS user_history (
        id SERIAL PRIMARY KEY,
        username VARCHAR(100) NOT NULL,
        action_type VARCHAR(50) NOT NULL,
        model_used VARCHAR(100),
        settings TEXT,
        input_snippet TEXT,
        output_snippet TEXT,
        language VARCHAR(50) DEFAULT 'English',
        word_count_in INTEGER DEFAULT 0,
        word_count_out INTEGER DEFAULT 0,
        char_count_in INTEGER DEFAULT 0,
        char_count_out INTEGER DEFAULT 0,
        created_at TIMESTAMP DEFAULT NOW(),
        FOREIGN KEY (username) REFERENCES users(username)
        ON DELETE CASCADE
        ON UPDATE CASCADE
    );
    """)

    # OTP TABLE
    cur.execute("""
    CREATE TABLE IF NOT EXISTS email_otps (
        email VARCHAR(150) PRIMARY KEY,
        otp VARCHAR(10),
        created_at TIMESTAMP DEFAULT NOW(),
        FOREIGN KEY (email) REFERENCES users(email)
        ON DELETE CASCADE
        ON UPDATE CASCADE
    );
    """)



    cur.execute("""
CREATE TABLE IF NOT EXISTS feedback (
    id SERIAL PRIMARY KEY,
    username VARCHAR(100),
    feature VARCHAR(50),
    rating INTEGER,
    feedback_text TEXT,
    created_at TIMESTAMP DEFAULT NOW()
);
""")

    conn.commit()
    cur.close()
    conn.close()

create_tables()

def save_feedback(username, feature, rating):

    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
        INSERT INTO feedback (username, feature, rating)
        VALUES (%s, %s, %s)
    """, (username, feature, rating))

    conn.commit()
    cur.close()
    conn.close()

# ── History helpers ───────────────────────────────────────
def save_history_to_db(username, action_type, model_used, settings,
                       input_text, output_text, language="English"):
    try:
        conn = get_connection(); cur = conn.cursor()
        inp  = (input_text[:300]  + "...") if len(input_text)  > 300 else input_text
        out  = (output_text[:300] + "...") if len(output_text) > 300 else output_text
        cur.execute("""INSERT INTO user_history
            (username,action_type,model_used,settings,input_snippet,output_snippet,
             language,word_count_in,word_count_out,char_count_in,char_count_out,created_at)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)""",
            (username, action_type, model_used, settings, inp, out, language,
             len(input_text.split()), len(output_text.split()),
             len(input_text), len(output_text),
             datetime.datetime.utcnow()))
        conn.commit(); cur.close(); conn.close()
    except Exception: pass

def get_history_from_db(username, limit=100):
    try:
        conn = get_connection(); cur = conn.cursor()
        cur.execute("""SELECT id,action_type,model_used,settings,input_snippet,
            output_snippet,language,word_count_in,word_count_out,
            COALESCE(char_count_in,0),COALESCE(char_count_out,0),created_at
            FROM user_history WHERE username=%s ORDER BY created_at DESC LIMIT %s""",
            (username, limit))
        data = cur.fetchall(); cur.close(); conn.close(); return data
    except: return []

def get_all_history_from_db(limit=200):
    try:
        conn = get_connection(); cur = conn.cursor()
        cur.execute("""SELECT username,action_type,model_used,settings,input_snippet,
            output_snippet,language,word_count_in,word_count_out,
            COALESCE(char_count_in,0),COALESCE(char_count_out,0),created_at
            FROM user_history ORDER BY created_at DESC LIMIT %s""", (limit,))
        data = cur.fetchall(); cur.close(); conn.close(); return data
    except: return []

def clear_user_history(username):
    try:
        conn = get_connection(); cur = conn.cursor()
        cur.execute("DELETE FROM user_history WHERE username=%s", (username,))
        conn.commit(); cur.close(); conn.close()
    except: pass

def update_user_email(username, new_email):

    conn = get_connection()
    cur = conn.cursor()

    try:

        # Get old email
        cur.execute(
            "SELECT email FROM users WHERE username=%s",
            (username,)
        )

        result = cur.fetchone()

        if not result:
            st.error("User not found")
            return False

        old_email = result[0]

        # Update parent table
        cur.execute(
            "UPDATE users SET email=%s WHERE username=%s",
            (new_email, username)
        )

        # No need to update child tables (password_history, email_otps) explicitly
        # if ON UPDATE CASCADE is set on foreign keys.

        conn.commit()

        return True

    except Exception as e:

        conn.rollback()
        st.error(f"Update error: {e}")
        return False

    finally:

        cur.close()
        conn.close()

def update_avatar(username, avatar_data):
    conn = get_connection()
    cur = conn.cursor()
    try:
        cur.execute("UPDATE users SET avatar=%s WHERE username=%s", (avatar_data, username))
        conn.commit()
    finally:
        cur.close()
        conn.close()


# ── Rate limiting ─────────────────────────────────────────
def get_login_attempts(email):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email=%s", (email,))
    data = cur.fetchone(); cur.close(); conn.close()
    return data if data else (0, 0)

def increment_login_attempts(email):
    conn = get_connection(); cur = conn.cursor()
    attempts, _ = get_login_attempts(email); now = time.time()
    cur.execute("""INSERT INTO login_attempts (email,attempts,last_attempt) VALUES (%s,%s,%s)
        ON CONFLICT (email) DO UPDATE SET attempts=%s, last_attempt=%s""",
        (email, attempts+1, now, attempts+1, now))
    conn.commit(); cur.close(); conn.close()

def reset_login_attempts(email):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("DELETE FROM login_attempts WHERE email=%s", (email,))
    conn.commit(); cur.close(); conn.close()

def is_rate_limited(email):
    attempts, last_attempt = get_login_attempts(email)
    if attempts >= MAX_LOGIN_ATTEMPTS:
        elapsed = time.time() - last_attempt
        if elapsed < LOCKOUT_SECONDS: return True, int(LOCKOUT_SECONDS - elapsed)
        else: reset_login_attempts(email)
    return False, 0
# ── OTP FUNCTIONS ─────────────────────────────────────────

def send_email_otp(email):

    otp = str(random.randint(100000,999999))

    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
    INSERT INTO email_otps(email, otp, created_at)
    VALUES(%s,%s,NOW())
    ON CONFLICT (email)
    DO UPDATE SET otp=%s, created_at=NOW()
    """,(email,otp,otp))

    conn.commit()
    cur.close()
    conn.close()

    try:

        msg = MIMEMultipart()
        msg["From"] = EMAIL_ADDRESS
        msg["To"] = email
        msg["Subject"] = "Your Login OTP"

        body = f"Your OTP is {otp}. It expires in {OTP_EXPIRY_MINUTES} minutes."

        msg.attach(MIMEText(body,"plain"))

        server = smtplib.SMTP("smtp.gmail.com",587)
        server.starttls()
        server.login(EMAIL_ADDRESS,EMAIL_PASSWORD)

        server.send_message(msg)
        server.quit()

        return True

    except Exception as e:
        st.error(f"Email error: {e}")
        return False

def verify_email_otp(email,otp):

    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
    SELECT otp, created_at FROM email_otps
    WHERE email=%s
    """,(email,))

    row = cur.fetchone()

    cur.close()
    conn.close()

    if not row:
        return False

    stored_otp, created = row

    expiry = created + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)

    if datetime.datetime.utcnow() > expiry:
        return False

    if otp == stored_otp:
        return True

    return False

# ── User management ───────────────────────────────────────
def check_user_exists_by_email(email):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT id FROM users WHERE email=%s", (email,))
    data = cur.fetchone(); cur.close(); conn.close(); return data is not None

def check_user_exists_by_username(username):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT id FROM users WHERE username=%s", (username,))
    data = cur.fetchone(); cur.close(); conn.close(); return data is not None

def register_user(username, email, password, security_question, security_answer):
    conn = get_connection(); cur = conn.cursor()
    try:
        hp = bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()
        cur.execute("INSERT INTO users (username,email,password,security_question,security_answer) VALUES (%s,%s,%s,%s,%s)",
            (username, email, hp, security_question, security_answer.strip().lower()))
        cur.execute("INSERT INTO password_history (email,password) VALUES (%s,%s)", (email, hp))
        conn.commit(); return True, "Success"
    except Exception as e: conn.rollback(); return False, str(e)
    finally: cur.close(); conn.close()

def authenticate_user(email, password):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT username, password, role, is_locked FROM users WHERE email=%s", (email,))
    result = cur.fetchone()
    if result:
      uname, hp, role, locked = result
      if locked:
        return "LOCKED", None, None

      if bcrypt.checkpw(password.encode(), hp.encode()):
        reset_login_attempts(email)
        return True, uname, role
    increment_login_attempts(email)
    return False, None, None

def check_password_reused(email, new_password):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT password FROM password_history WHERE email=%s ORDER BY set_at DESC", (email,))
    for (hp,) in cur.fetchall():
        if bcrypt.checkpw(new_password.encode(), hp.encode()): cur.close(); conn.close(); return True
    cur.close(); conn.close(); return False

def check_is_old_password(email, password):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT password,set_at FROM password_history WHERE email=%s ORDER BY set_at DESC", (email,))
    for hp, set_at in cur.fetchall():
        if bcrypt.checkpw(password.encode(), hp.encode()): cur.close(); conn.close(); return set_at
    cur.close(); conn.close(); return None

def update_password(email, new_password):
    conn = get_connection(); cur = conn.cursor()
    hp = bcrypt.hashpw(new_password.encode(), bcrypt.gensalt()).decode()
    cur.execute("UPDATE users SET password=%s WHERE email=%s", (hp, email))
    cur.execute("INSERT INTO password_history (email,password) VALUES (%s,%s)", (email, hp))
    conn.commit(); cur.close(); conn.close()

def get_security_question(email):
    conn = get_connection(); cur = conn.cursor()
    cur.execute("SELECT security_question,security_answer FROM users WHERE email=%s", (email,))
    result = cur.fetchone(); cur.close(); conn.close(); return result

def get_all_users():
    conn = get_connection()
    cur = conn.cursor()

    cur.execute("""
        SELECT username, email, role, is_locked, created_at
        FROM users
        ORDER BY created_at DESC
    """)

    data = cur.fetchall()
    cur.close()
    conn.close()

    return data

def delete_user(email):

    conn = get_connection()
    cur = conn.cursor()

    cur.execute("DELETE FROM users WHERE email=%s", (email,))

    conn.commit()

    cur.close()
    conn.close()

def get_user_profile(username):

    conn = get_connection()
    cur = conn.cursor()

    cur.execute(
        "SELECT email, avatar FROM users WHERE username=%s",
        (username,)
    )

    result = cur.fetchone()

    cur.close()
    conn.close()

    if result:
        return result[0], result[1]

    return None, None


def update_user_avatar(username, avatar):

    conn = get_connection()
    cur = conn.cursor()

    try:
        cur.execute(
            "UPDATE users SET avatar=%s WHERE username=%s",
            (avatar, username)
        )

        conn.commit()

    finally:
        cur.close()
        conn.close()
if st.session_state.get("otp_stage"):

    st.subheader("Enter OTP")

    otp = st.text_input("OTP")

    if st.button("Verify OTP"):

        email = st.session_state["otp_email"]

        if verify_email_otp(email, otp):

            conn = get_connection()
            cur = conn.cursor()

            cur.execute(
                "SELECT username FROM users WHERE email=%s",
                (email,)
            )

            user = cur.fetchone()

            cur.close()
            conn.close()

            st.session_state["user"] = user[0]

            st.success("Login successful")

            st.rerun()

        else:

            st.error("Invalid or expired OTP")
# ── JWT ───────────────────────────────────────────────────
def create_access_token(data, expires_minutes=ACCESS_TOKEN_EXPIRE_MINUTES):
    d = data.copy()
    d["exp"] = datetime.datetime.utcnow() + datetime.timedelta(minutes=expires_minutes)
    return jwt.encode(d, SECRET_KEY, algorithm=ALGORITHM)

def verify_token(token):
    try: return jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
    except: return None

# ── Validation ────────────────────────────────────────────
def is_valid_email(email):
    return re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', email) is not None

def check_password_strength(password):
    has_upper   = bool(re.search(r"[A-Z]", password))
    has_lower   = bool(re.search(r"[a-z]", password))
    has_digit   = bool(re.search(r"\d",    password))
    has_special = bool(re.search(r'[!@#$%^&*(),.?":{}|<>]', password))
    has_space   = bool(re.search(r"\s",    password))
    if has_space: return "Weak", ["No spaces allowed"]
    is_alphanum = (has_upper or has_lower) and has_digit
    if len(password) >= 8 and is_alphanum and has_special: return "Strong", []
    if len(password) >= 8 and is_alphanum: return "Medium", ["Add special characters"]
    return "Weak", ["Min 8 chars with letters and numbers"]

def get_relative_time(dt):
    if not dt: return "a while ago"
    try:
        if isinstance(dt, str): dt = datetime.datetime.strptime(dt, "%Y-%m-%d %H:%M:%S")
        # Strip tz if present (DB returns UTC naive or aware), compare with utcnow
        if hasattr(dt, 'tzinfo') and dt.tzinfo is not None:
            dt = dt.replace(tzinfo=None)
        diff = datetime.datetime.utcnow() - dt
        total_seconds = int(diff.total_seconds())
        if total_seconds < 60:   return "just now"
        elif total_seconds < 3600: return f"{total_seconds // 60}m ago"
        elif total_seconds < 86400: return f"{total_seconds // 3600}h ago"
        elif diff.days > 365: return f"{diff.days//365}y ago"
        elif diff.days > 30:  return f"{diff.days//30}mo ago"
        else: return f"{diff.days}d ago"
    except: return str(dt)

# ── OTP ───────────────────────────────────────────────────
def generate_otp():
    secret    = secrets.token_bytes(20)
    msg       = struct.pack(">Q", int(time.time()))
    h         = hmac.new(secret, msg, hashlib.sha1).digest()
    offset    = h[19] & 0xf
    code      = ((h[offset] & 0x7f) << 24 | (h[offset+1] & 0xff) << 16 |
                 (h[offset+2] & 0xff) << 8  | (h[offset+3] & 0xff))
    return f"{code % 1000000:06d}"

def create_otp_token(otp, email):
    otp_hash = bcrypt.hashpw(otp.encode(), bcrypt.gensalt()).decode()
    payload  = {'otp_hash': otp_hash, 'sub': email, 'type': 'password_reset',
                'iat': datetime.datetime.utcnow(),
                'exp': datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)}
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        if payload.get('sub') != email: return False, "Token mismatch"
        if bcrypt.checkpw(input_otp.encode(), payload['otp_hash'].encode()): return True, "Valid"
        return False, "Invalid OTP"
    except Exception as e: return False, str(e)

def send_otp_email(to_email, otp):
    msg = MIMEMultipart()
    msg['From'] = f"LLM S <{EMAIL_ADDRESS}>"
    msg['To'] = to_email
    msg['Subject'] = "Your Password Reset OTP"
    body = f"""<html><body style=\"background:#0d0a0a;font-family:Inter,sans-serif;padding:40px;text-align:center;\">
    <div style=\"background: linear-gradient(135deg,#ff6ec7,#845ec2,#2c73d2,#00c9a7);border-radius:16px;padding:40px;max-width:480px;margin:auto;border:1px solid rgba(192,57,43,0.3);\">
        <div style=\"font-size:48px;margin-bottom:16px;\">🔐</div>
        <h2 style=\"color:#ff6b6b;margin:0 0 12px;\">Password Reset OTP</h2>
        <p style=\"color:#c09090;margin-bottom:24px;\">Use this code for <strong style=\"color:#ff8a8a;\">{to_email}</strong></p>
        <div style=\"background:#200f0f;color:#ff6b6b;font-size:36px;font-weight:700;letter-spacing:10px;
                    padding:24px;border-radius:12px;margin:0 0 24px;border:1px solid rgba(192,57,43,0.4);\">{otp}</div>
        <p style=\"color:#7a5050;font-size:13px;\">Valid for {OTP_EXPIRY_MINUTES} minutes. Do not share this code.</p>
    </div></body></html>"""
    msg.attach(MIMEText(body, 'html'))
    try:
        s = smtplib.SMTP('smtp.gmail.com', 587)
        s.starttls(); s.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        s.sendmail(EMAIL_ADDRESS, to_email, msg.as_string()); s.quit()
        return True, "Sent"
    except Exception as e: return False, str(e)

# ── Session init ──────────────────────────────────────────
for key, val in [("jwt_token", None), ("page", "login"), ("username", None)]:
    if key not in st.session_state: st.session_state[key] = val

# ── Helpers ───────────────────────────────────────────────
def page_header(icon, title, subtitle=""):
    sub_html = f'<div class="ph-sub">{subtitle}</div>' if subtitle else ""
    st.markdown(f"""<div class="page-header">
        <div class="ph-icon">{icon}</div>
        <div><div class="ph-title">{title}</div>{sub_html}</div>
    </div>""", unsafe_allow_html=True)

def create_gauge(value, title, min_val=0, max_val=100, color="#e74c3c"):

    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=value,

        title={'text': title, 'font': {'color': "white", 'size': 13}},

        number={'font': {'color': "white", 'size': 18}},

        gauge={
            'axis': {
                'range': [min_val, max_val],
                'tickwidth': 1,
                'tickcolor': "white"
            },

            'bar': {'color': color},

            'bgcolor': "rgba(255,255,255,0.05)",

            'borderwidth': 2,
            'bordercolor': "rgba(255,255,255,0.25)",

            'steps': [
                {'range': [min_val, max_val], 'color': "rgba(255,255,255,0.08)"}
            ],
        }
    ))

    fig.update_layout(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font={'color': "white", 'family': "Inter"},
        height=230,
        margin=dict(l=10, r=10, t=40, b=10)
    )

    return fig

def promote_user(email):

    conn = get_connection()
    cur = conn.cursor()

    cur.execute("UPDATE users SET role='admin' WHERE email=%s",(email,))

    conn.commit()
    cur.close()
    conn.close()


def demote_user(email):

    conn = get_connection()
    cur = conn.cursor()

    cur.execute("UPDATE users SET role='user' WHERE email=%s",(email,))

    conn.commit()
    cur.close()
    conn.close()


def lock_user(email):

    conn = get_connection()
    cur = conn.cursor()

    cur.execute("UPDATE users SET is_locked=TRUE WHERE email=%s",(email,))

    conn.commit()
    cur.close()
    conn.close()


def unlock_user(email):

    conn = get_connection()
    cur = conn.cursor()

    cur.execute("UPDATE users SET is_locked=FALSE WHERE email=%s",(email,))

    conn.commit()
    cur.close()
    conn.close()


def generate_wordcloud(text_data):

    # Load mask image (make sure mask.png is in same folder)
    try:
        mask = np.array(Image.open("mask.png"))
    except:
        mask = None   # fallback if no image

    wc = WordCloud(
        width=1000,
        height=500,
        background_color="white",
        colormap="Set2",
        mask=mask,
        contour_width=2,
        contour_color='black'
    ).generate(" ".join(text_data))

    fig, ax = plt.subplots(figsize=(10,5))
    ax.imshow(wc, interpolation='bilinear')
    ax.axis("off")

    return fig

def get_feedback_data():

    conn = get_connection()
    cur = conn.cursor()

    try:
        cur.execute("SELECT rating FROM feedback")
        data = cur.fetchall()
        return [r[0] for r in data]

    except Exception:
        return []

    finally:
        cur.close()
        conn.close()


def rating_to_words(ratings):

    words = []

    for r in ratings:
        if r == 5:
            words += ["excellent"] * 5
        elif r == 4:
            words += ["good"] * 4
        elif r == 3:
            words += ["average"] * 3
        elif r == 2:
            words += ["poor"] * 2
        else:
            words += ["bad"]

    return words

# ══════════════════════════════════════════════════════════
# AUTH PAGES
# ══════════════════════════════════════════════════════════
def admin_dashboard():

    token = st.session_state["jwt_token"]
    payload = verify_token(token)

    if not payload:
        st.session_state["jwt_token"] = None
        st.session_state["page"] = "login"
        st.rerun()
        return

    username = payload.get("username")
    email = payload.get("sub")

    # ── HEADER ─────────────────────────────

    page_header("🛡️", "Admin Dashboard",
                "Management & Analytics Control Center")

    # ── ADMIN PROFILE BAR ─────────────────
    st.markdown(f"""
    <div class="page-header">
    <div style="font-size:18px;font-weight:600;">👤 {username}</div>
    <div style="font-size:13px;opacity:0.7;">{email}</div>
    </div>
    """, unsafe_allow_html=True)

    # ── METRICS ───────────────────────────
    users = get_all_users()
    history = get_all_history_from_db(500)

    total_users = len(users)
    locked_users = sum(1 for u in users if u[3])
    admins = sum(1 for u in users if u[2] == "admin")

    m1,m2,m3 = st.columns(3)

    m1.metric("👥 Total Users", total_users)
    m2.metric("🔒 Locked Accounts", locked_users)
    m3.metric("🛡️ Admins", admins)

    st.markdown("---")

    # ──────────────────────────────────────
    # USER MANAGEMENT
    # ──────────────────────────────────────

    st.markdown("### 👥 User Management")

    for uname, email, role, locked, created in users:

        c1,c2,c3,c4,c5,c6 = st.columns([2,3,1,1,1,1])

        c1.markdown(f"**{uname}**")
        c2.markdown(email)

        role_color = "#4ade80" if role == "admin" else "#60a5fa"
        c3.markdown(
            f"<span style='color:{role_color};font-weight:600'>{role}</span>",
            unsafe_allow_html=True
        )

        c4.markdown("🔒" if locked else "🔓")

        # Promote/Demote

        if role != "admin":
            if c5.button("Promote", key=f"pro_{email}"):
                promote_user(email)
                st.rerun()
        else:
            if c5.button("Demote", key=f"dem_{email}"):
                demote_user(email)
                st.rerun()

        # Lock / Unlock

        if locked:
            if c6.button("Unlock", key=f"unlock_{email}"):
                unlock_user(email)
                st.rerun()
        else:
            if c6.button("Lock", key=f"lock_{email}"):
                lock_user(email)
                st.rerun()

        if st.button("Delete User", key=f"del_{email}"):
            delete_user(email)
            st.rerun()

    st.markdown("---")

    # ──────────────────────────────────────
    # ACTIVITY TRACKING
    # ──────────────────────────────────────

    st.markdown("### 📊 System Activity")

    from collections import Counter

    actions = Counter(h[1] for h in history)

    fig = go.Figure(go.Bar(
        x=list(actions.keys()),
        y=list(actions.values()),
        marker_color=["#00c9a7","#845ec2","#2c73d2"]
    ))

    fig.update_layout(
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        font=dict(color="white"),
        title="Platform Usage"
    )

    st.plotly_chart(fig, use_container_width=True)

    st.markdown("---")

        # ──────────────────────────────────────
    # ADVANCED ANALYTICS
    # ──────────────────────────────────────

    st.markdown("### 📊 Advanced Analytics")

    # Model usage
    model_counts = Counter(h[2] for h in history if h[2])

    fig_model = go.Figure(go.Bar(
        x=list(model_counts.keys()),
        y=list(model_counts.values()),
        marker_color="#845ec2"
    ))

    fig_model.update_layout(title="Model Usage", font=dict(color="white"))
    st.plotly_chart(fig_model, use_container_width=True)

    # Language usage
    lang_counts = Counter(h[6] for h in history if h[6])

    fig_lang = go.Figure(go.Pie(
        labels=list(lang_counts.keys()),
        values=list(lang_counts.values())
    ))

    fig_lang.update_layout(title="Language Usage", font=dict(color="white"))
    st.plotly_chart(fig_lang, use_container_width=True)

    # Feature usage
    feature_counts = Counter(h[1] for h in history)

    fig_feat = go.Figure(go.Bar(
        x=list(feature_counts.keys()),
        y=list(feature_counts.values()),
        marker_color="#00c9a7"
    ))

    fig_feat.update_layout(title="Feature Usage", font=dict(color="white"))
    st.plotly_chart(fig_feat, use_container_width=True)

    # ──────────────────────────────────────
    # WORD CLOUD
    # ──────────────────────────────────────



    st.markdown("### ☁️ Feedback Insights")
    ratings = get_feedback_data()
    if ratings:
        words = rating_to_words(ratings)
        fig = generate_wordcloud(words)

        st.pyplot(fig)

    else:
        st.info("No feedback available")

    # ──────────────────────────────────────
    # EXPORT DATA
    # ──────────────────────────────────────

    st.markdown("### 📥 Export Data")

    df = pd.DataFrame(history, columns=[
        "username","action","model","settings",
        "input","output","language",
        "wc_in","wc_out","cc_in","cc_out","time"
    ])

    csv = df.to_csv(index=False).encode("utf-8")

    st.download_button(
        "Download Full Data",
        csv,
        "admin_data.csv",
        "text/csv"
    )


    # ──────────────────────────────────────
    # REAL TIME USERS
    # ──────────────────────────────────────

    st.markdown("### 🟢 Active Users")

    import random

    if total_users > 0:
        active_users = random.randint(1, total_users)
        st.success(f"{active_users} users currently active")
    else:
         t.warning("No users available yet")


    if st.button("🔓 Logout", type="secondary"):

        st.session_state["jwt_token"] = None
        st.session_state["page"] = "login"

        st.rerun()

def signup_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 1.4, 1])
    with col2:
        st.markdown("""<div style="text-align:center;margin-bottom:32px;">
            <div style="font-size:26px;font-weight:700;color:#ff6b6b;margin-top:8px;">Create Account</div>
            <div style="font-size:13px;color:#7a5050;margin-top:4px;">Join Infosys LLM Studio</div>
        </div>""", unsafe_allow_html=True)
        with st.container():
            username = st.text_input("👤  Username")
            email    = st.text_input("📧  Email")
            password = st.text_input("🔑  Password", type="password")
            if password:
                strength, feedback = check_password_strength(password)
                color = {"Strong":"#ff6b6b","Medium":"#ff9944","Weak":"#ff4444"}[strength]
                st.markdown(f'<div style="margin:4px 0 8px;font-size:13px;">Strength: <span style="color:{color};font-weight:600;">{strength}</span></div>', unsafe_allow_html=True)
                if feedback: st.caption(f"💡 {', '.join(feedback)}")
            confirm_password = st.text_input("🔑  Confirm Password", type="password")
            question = st.selectbox("🛡️  Security Question", SECURITY_QUESTIONS)
            answer   = st.text_input("💬  Security Answer")
            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("✨  Create Account", type="primary"):
                username = username.strip(); email = email.strip().lower()
                answer_clean = answer.strip().lower()
                errors = []
                if not username: errors.append("Username required")
                if not is_valid_email(email): errors.append("Valid email required")
                if not password: errors.append("Password required")
                else:
                    s, fb = check_password_strength(password)
                    if s == "Weak": errors.append(f"Weak password: {', '.join(fb)}")
                if password != confirm_password: errors.append("Passwords don't match")
                if not answer_clean: errors.append("Security answer required")
                if errors:
                    for e in errors: st.error(e)
                    return
                if check_user_exists_by_email(email): st.error("Email already registered"); return
                if check_user_exists_by_username(username): st.error("Username taken"); return
                ok, msg = register_user(username, email, password, question, answer_clean)
                if ok: st.success("✅ Account created!"); time.sleep(1); st.session_state["page"]="login"; st.rerun()
                else: st.error(f"Error: {msg}")
            if st.button("← Back to Login", type="secondary"):
                st.session_state["page"]="login"; st.rerun()

def login_page():
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1, 1.4, 1])
    with col2:
        st.markdown("""
<div style="text-align:center;margin-bottom:32px;">

<img src="https://play-lh.googleusercontent.com/Fjga-fnKPNPSg1_3mVis70od3LrCSvrL1lmwl2123h_RJ4DxlUjCQLym05LqDapG2Q"
     style="
     width:70px;
     height:70px;
     border-radius:16px;
     padding:6px;
     background:rgba(255,255,255,0.15);
     backdrop-filter:blur(10px);
     box-shadow:0 4px 18px rgba(0,0,0,0.35);
     margin-bottom:10px;
     ">

<div style="font-size:26px;font-weight:700;color:white;margin-top:8px;">
    Infosys Login
</div>

<div style="font-size:13px;color:rgba(255,255,255,0.75);margin-top:4px;">
    Sign in to Infosys LLM Studio
</div>

</div>
""", unsafe_allow_html=True)
        email    = st.text_input("📧  Email")
        password = st.text_input("🔑  Password", type="password")
        st.markdown("<br>", unsafe_allow_html=True)
        if st.button("🚀  Sign In", type="primary"):
            if not email or not password: st.error("All fields required"); return
            is_locked, wait_time = is_rate_limited(email)
            if is_locked: st.error(f"🔒 Account locked — try again in {wait_time}s"); return
            if email == ADMIN_EMAIL_ID and password == ADMIN_PASSWORD:
                token = create_access_token({
                    "sub": email,
                    "username": "admin",
                    "role": "admin"
                    })
                st.session_state.update({"jwt_token":token,"username":"admin","nav_selected":"Readability"})
                st.success("✅ Admin access granted!"); time.sleep(1); st.rerun()
            success, uname, role = authenticate_user(email, password)
            if success == True:
              token = create_access_token({
                  "sub": email,
                  "username": uname,
                  "role": role
              })
              st.session_state.update({
                  "jwt_token": token,
                  "username": uname,
                  "role": role
              })

              st.success(f"✅ Welcome back, {uname}!")
              time.sleep(1)
              st.rerun()
            elif success == "LOCKED":
              st.error("🔒 Your account is locked by admin")
            else:
              st.error("Invalid credentials")
              old_dt = check_is_old_password(email, password)
              if old_dt: st.warning(f"⚠️ This was a previously used password ({get_relative_time(old_dt)})")
              attempts, _ = get_login_attempts(email)
              remaining = MAX_LOGIN_ATTEMPTS - attempts
              if remaining > 0: st.caption(f"⚠️ {remaining} attempt(s) left before lockout")
        ca, cb = st.columns(2)
        with ca:
            if st.button("📝  Register", type="secondary"):
                st.session_state["page"]="signup"; st.rerun()
        with cb:
            if st.button("🔓  Forgot Password", type="secondary"):
                st.session_state["page"]="forgot"; st.rerun()

def forgot_password_page():

    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2, col3 = st.columns([1,1.4,1])

    with col2:

        st.markdown("""
        <div style="text-align:center;margin-bottom:24px;">
        <div style="font-size:48px">🔓</div>
        <div style="font-size:24px;font-weight:700;color:#ff6b6b;margin-top:8px;">
        Reset Password
        </div>
        </div>
        """, unsafe_allow_html=True)

        if "forgot_stage" not in st.session_state:
            st.session_state["forgot_stage"] = "email"

        stage = st.session_state["forgot_stage"]

# ---------------- EMAIL ----------------

        if stage == "email":

            email_input = st.text_input("📧 Registered Email")

            if st.button("Next →", type="primary"):

                if not is_valid_email(email_input):
                    st.error("Enter valid email")

                elif not check_user_exists_by_email(email_input):
                    st.error("Email not registered")

                else:
                    st.session_state["reset_email"] = email_input.strip().lower()
                    st.session_state["forgot_stage"] = "method"
                    st.rerun()

# ---------------- METHOD SELECTION ----------------

        elif stage == "method":

            st.markdown("### Choose Verification Method")

            colA, colB = st.columns(2)

            with colA:
                if st.button("📨 Verify with OTP", use_container_width=True):
                    st.session_state["forgot_stage"] = "otp_send"
                    st.rerun()

            with colB:
                if st.button("🛡️ Security Question", use_container_width=True):
                    st.session_state["forgot_stage"] = "security"
                    st.rerun()

# ---------------- SEND OTP ----------------

        elif stage == "otp_send":

            email = st.session_state["reset_email"]

            st.success(f"OTP will be sent to **{email}**")

            if st.button("Send OTP", type="primary"):

                otp = generate_otp()

                ok, msg = send_otp_email(email, otp)

                if ok:
                    st.session_state["otp_token"] = create_otp_token(otp, email)
                    st.session_state["otp_start_time"] = time.time()
                    st.session_state["forgot_stage"] = "otp_verify"
                    st.success("OTP sent!")
                    st.rerun()

                else:
                    st.error(msg)
# ---------------- VERIFY OTP ----------------
        elif stage == "otp_verify":
              email = st.session_state["reset_email"]

              remaining = OTP_EXPIRY_MINUTES*60 - (time.time() - st.session_state["otp_start_time"])

              if remaining > 0:
                  st.info(f"OTP expires in {int(remaining)} seconds")
              else:
                  st.error("OTP expired. Please click Resend OTP.")

              otp_input = st.text_input("Enter OTP", max_chars=6)

              if st.button("Resend OTP"):

                  otp = generate_otp()
                  ok, msg = send_otp_email(email, otp)

                  if ok:
                      st.session_state["otp_token"] = create_otp_token(otp, email)
                      st.session_state["otp_start_time"] = time.time()
                      st.success("New OTP sent to your email")
                  else:
                      st.error(msg)

              if st.button("Verify OTP"):

                  if remaining <= 0:
                     st.error("OTP expired. Please resend OTP.")
                     return

                  if not otp_input:
                     st.error("Please enter the OTP")
                     return

                  ok, msg = verify_otp_token(
                  st.session_state.get("otp_token", ""),
                  otp_input,
                  email
                 )

                  if ok:
                    st.session_state["forgot_stage"] = "reset"
                    st.rerun()
                  else:
                    st.error(msg)

# ---------------- RESET PASSWORD ----------------

        elif stage == "reset":

            email = st.session_state["reset_email"]

            new_password = st.text_input("New Password", type="password")
            confirm_password = st.text_input("Confirm Password", type="password")

            if st.button("Update Password", type="primary"):

                if new_password != confirm_password:
                    st.error("Passwords do not match")

                elif check_password_reused(email, new_password):
                    st.error("Cannot reuse previous password")

                else:

                    update_password(email, new_password)

                    st.success("Password updated successfully")

                    for k in ["forgot_stage","reset_email","otp_token"]:
                        st.session_state.pop(k, None)

                    time.sleep(1)
                    st.session_state["page"] = "login"
                    st.rerun()

        st.markdown("---")

        if st.button("← Back to Login"):

            for k in ["forgot_stage","reset_email","otp_token"]:
                st.session_state.pop(k, None)

            st.session_state["page"] = "login"
            st.rerun()
# ══════════════════════════════════════════════════════════
# DASHBOARD PAGES
# ══════════════════════════════════════════════════════════
def readability_page():
    username = st.session_state.get("username","")
    page_header("📊", "Text Readability Analyzer", "Measure how easy your text is to understand")
    tab1, tab2 = st.tabs(["✍️  Type / Paste Text", "📂  Upload File"])
    text_input = ""
    with tab1:
        raw_text = st.text_area("Paste your text here (min 50 characters):", height=180, label_visibility="collapsed",
                                placeholder="Paste your text here...")
        if raw_text: text_input = raw_text
    with tab2:
        try:
            uploaded_file = st.file_uploader("Upload a .txt or .pdf file", type=["txt","pdf"],
                                             label_visibility="collapsed")
            if uploaded_file:
                if uploaded_file.type == "application/pdf":
                    reader = PyPDF2.PdfReader(uploaded_file)
                    text_input = "".join([p.extract_text()+"\n" for p in reader.pages])
                    st.success(f"✅  Loaded {len(reader.pages)} page(s) from PDF")
                else:
                    text_input = uploaded_file.read().decode("utf-8")
                    st.success(f"✅  Loaded: {uploaded_file.name}")
        except Exception as e: st.error(f"Error: {e}")
    st.markdown("<br>", unsafe_allow_html=True)
    if st.button("🔍  Analyze Readability", type="primary"):
        if len(text_input) < 50: st.error("Text too short — minimum 50 characters"); return
        with st.spinner("Analyzing text..."):
            analyzer = ReadabilityAnalyzer(text_input)
            scores   = analyzer.get_all_metrics()
        avg_grade = (scores['Flesch-Kincaid Grade'] + scores['Gunning Fog'] +
                     scores['SMOG Index'] + scores['Coleman-Liau']) / 4
        if   avg_grade <= 6:  level, lcolor = "Beginner",     "#4ade80"
        elif avg_grade <= 10: level, lcolor = "Intermediate", "#60a5fa"
        elif avg_grade <= 14: level, lcolor = "Advanced",     "#fb923c"
        else:                 level, lcolor = "Expert",       "#f87171"
        st.markdown(f"""<div style="background:rgba(192,57,43,0.08);border:1px solid rgba(192,57,43,0.25);
            border-left:5px solid {lcolor};border-radius:12px;padding:20px 24px;margin:20px 0;
            display:flex;align-items:center;gap:20px;">
            <div style="font-size:40px;">📚</div>
            <div>
                <div style="font-size:20px;font-weight:700;color:{lcolor};">{level} Level</div>
                <div style="color:#9a7070;font-size:13px;margin-top:3px;">Approx. Grade {int(avg_grade)} · Flesch {round(scores['Flesch Reading Ease'],1)}</div>
            </div>
        </div>""", unsafe_allow_html=True)
        c1,c2,c3,c4,c5 = st.columns(5)
        for col, label, val in zip([c1,c2,c3,c4,c5],
            ["Sentences","Words","Syllables","Complex","Characters"],
            [analyzer.num_sentences,analyzer.num_words,analyzer.num_syllables,
             analyzer.complex_words,analyzer.char_count]):
            col.metric(label, val)
        st.markdown("<br>", unsafe_allow_html=True)
        gc1,gc2,gc3 = st.columns(3)
        gauge_data = [
            ("Flesch Reading Ease", 0, 100, "#e74c3c", gc1),
            ("Flesch-Kincaid Grade", 0, 20, "#c0392b", gc2),
            ("SMOG Index", 0, 20, "#a93226", gc3),
        ]
        gc4,gc5 = st.columns(2)
        gauge_data += [("Gunning Fog",0,20,"#922b21",gc4), ("Coleman-Liau",0,20,"#7b241c",gc5)]
        for metric, mn, mx, clr, col in gauge_data:
            with col: st.plotly_chart(create_gauge(scores[metric],metric,mn,mx,clr), use_container_width=True)
        names  = list(scores.keys())
        vals   = list(scores.values())
        bar_fig = go.Figure(go.Bar(
            x=names, y=vals,
            marker_color=["#e74c3c","#c0392b","#a93226","#922b21","#7b241c"],
            text=[f"{v:.1f}" for v in vals], textposition="outside",
            textfont=dict(color="white",size=12)
        ))
        bar_fig.update_layout(
            paper_bgcolor="rgba(0,0,0,0)",plot_bgcolor="rgba(0,0,0,0)",
            font=dict(color="white",family="Inter"),
            xaxis=dict(tickfont=dict(color="#c07070"),gridcolor="#2a1010"),
            yaxis=dict(tickfont=dict(color="#c07070"),gridcolor="#2a1010"),
            title=dict(text="Readability Score Overview",font=dict(color="#ff6b6b",size=15),x=0.5),
            margin=dict(l=20,r=20,t=50,b=20), height=360
        )
        st.plotly_chart(bar_fig, use_container_width=True)
        st.dataframe(pd.DataFrame({
            "Metric": names,
            "Score":  [round(v,2) for v in vals],
            "Interpretation": [
                f"{'Easy' if scores['Flesch Reading Ease']>=60 else 'Challenging'} to read",
                f"Grade {int(scores['Flesch-Kincaid Grade'])} level",
                f"Grade {int(scores['SMOG Index'])} level",
                f"Grade {int(scores['Gunning Fog'])} level",
                f"Grade {int(scores['Coleman-Liau'])} level",
            ]
        }), use_container_width=True, hide_index=True)
        save_history_to_db(username,"Readability","textstat",
            f"Level:{level}|Grade:{int(avg_grade)}",text_input,
            f"Grade {int(avg_grade)} | {level} | Flesch={round(scores['Flesch Reading Ease'],1)}","English")

        st.markdown("### ⭐ Rate this result")

        rating = st.radio(
           "Your Feedback",
           [1,2,3,4,5],
        horizontal=True
       )

        if st.button("Submit Feedback", key="read_fb"):
             save_feedback(username, "Readability", rating)
             st.success("Thanks for your feedback!")


def summarizer_page():
    username = st.session_state.get("username","")
    page_header("📝", "Text Summarizer", "Condense long text using state-of-the-art AI models")
    if 'summarization_history' not in st.session_state: st.session_state.summarization_history = []
    col1, col2 = st.columns([2, 1])
    with col1:
        st.markdown("**Input Text**")
        text_input = st.text_area("", height=200, key="sum_text",
                                  placeholder="Paste your text here (min 50 characters)...",
                                  label_visibility="collapsed")
        uploaded_file = st.file_uploader("Or upload a file", type=["txt","pdf"], key="sum_upload")
        if uploaded_file:
            if uploaded_file.type == "application/pdf":
                reader = PyPDF2.PdfReader(uploaded_file)
                text_input = "".join([p.extract_text()+"\n" for p in reader.pages])
            else: text_input = uploaded_file.read().decode("utf-8")
            st.success(f"✅  Loaded ({len(text_input.split())} words)")
    with col2:
        st.markdown('<div class="settings-panel">', unsafe_allow_html=True)
        st.markdown("**⚙️ Settings**")
        summary_length = st.selectbox("📏  Length", ["Short","Medium","Long"])
        model_type     = st.selectbox("🤖  Model",  ["FLAN-T5","BART","Pegasus"])
        target_lang    = st.selectbox("🌐  Language", SUPPORTED_LANGUAGES)
        st.markdown("</div>", unsafe_allow_html=True)
        st.markdown("<br>", unsafe_allow_html=True)
        if st.button("▶  Generate Summary", type="primary", use_container_width=True):
            if len(text_input) < 50: st.error("Text too short (min 50 chars)")
            else:
                with st.spinner("Generating summary..."):
                    summary = engine.local_summarize(text_input, summary_length, model_type,
                                                     st.session_state.summarization_models, target_lang=target_lang)
                    if target_lang != "English":
                        with st.spinner(f"Translating to {target_lang}..."):
                            summary = engine.translate_text(summary, target_lang, st.session_state.translation_model)
                st.session_state.update({"last_summary":summary,"last_summary_text":text_input,"last_summary_lang":target_lang})
                st.session_state.summarization_history.append({
                    'timestamp': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    'input': text_input[:100]+("..." if len(text_input)>100 else ""),
                    'summary': summary, 'length': summary_length, 'model': model_type, 'lang': target_lang
                })
                save_history_to_db(username,"Summarize",model_type,
                    f"Length:{summary_length}|Lang:{target_lang}",text_input,summary,target_lang)
    if 'last_summary' in st.session_state:
        st.markdown("---")
        if st.session_state.get('last_summary_lang','English') != 'English':
            st.info(f"🌐  Output in **{st.session_state.last_summary_lang}**")
        r1, r2 = st.columns(2)
        with r1:
            st.markdown("**📄 Original**")
            st.markdown(f'<div class="result-box-original">{st.session_state.last_summary_text}</div>', unsafe_allow_html=True)
            st.caption(f"📊 {len(st.session_state.last_summary_text.split())} words")
        with r2:
            st.markdown("**✨ Summary**")
            st.markdown(f'<div class="result-box-output">{st.session_state.last_summary}</div>', unsafe_allow_html=True)
            st.caption(f"📊 {len(st.session_state.last_summary.split())} words")

            st.markdown("### ⭐ Rate this summary")

            rating = st.radio(
              "Your Feedback",
               [1,2,3,4,5],
            horizontal=True,
            key="sum_feedback_main"
            )

            if st.button("Submit Feedback", key="sum_feedback_btn"):
                  save_feedback(username, "Summarize", rating)
                  st.success("Feedback saved!")

        with st.expander("📜  Session History"):
            for item in reversed(st.session_state.summarization_history[-5:]):
                st.markdown(f"**{item['timestamp']}** — `{item['length']}` via `{item['model']}`")
                st.info(item['input']); st.success(item['summary']);
                st.markdown("---")


def paraphraser_page():
    username = st.session_state.get("username","")
    page_header("🔄", "Paraphrase Engine", "Rewrite text with different words and styles using AI")
    if 'paraphrasing_history' not in st.session_state: st.session_state.paraphrasing_history = []
    col1, col2 = st.columns([2, 1])
    with col1:
        st.markdown("**Input Text**")
        text_input = st.text_area("", height=200, key="para_text",
                                  placeholder="Paste your text here (min 50 characters)...",
                                  label_visibility="collapsed")
        uploaded_file = st.file_uploader("Or upload a file", type=["txt","pdf"], key="para_upload")
        if uploaded_file:
            if uploaded_file.type == "application/pdf":
                reader = PyPDF2.PdfReader(uploaded_file)
                text_input = "".join([p.extract_text()+"\n" for p in reader.pages])
            else: text_input = uploaded_file.read().decode("utf-8")
            st.success(f"✅  Loaded ({len(text_input.split())} words)")
    with col2:
        st.markdown('<div class="settings-panel">', unsafe_allow_html=True)
        st.markdown("**⚙️ Settings**")
        complexity  = st.selectbox("🎚️  Complexity", ["Simple","Neutral","Advanced"])
        style       = st.selectbox("🎨  Style",      ["Simplification","Formalization","Creative"])
        model_type  = st.selectbox("🤖  Model",      ["FLAN-T5","BART"], key="para_model")
        target_lang = st.selectbox("🌐  Language",   SUPPORTED_LANGUAGES, key="para_lang")
        st.markdown("</div>", unsafe_allow_html=True)
        st.markdown("<br>", unsafe_allow_html=True)
        if st.button("▶  Generate Paraphrase", type="primary", use_container_width=True):
            if len(text_input) < 50: st.error("Text too short (min 50 chars)")
            else:
                with st.spinner("Generating paraphrase..."):
                    para = engine.paraphrase_with_model(text_input, complexity, style, model_type,
                                                        st.session_state.paraphrase_models, target_lang=target_lang)
                    if target_lang != "English":
                        with st.spinner(f"Translating to {target_lang}..."):
                            para = engine.translate_text(para, target_lang, st.session_state.translation_model)
                st.session_state.update({"last_para":para,"last_para_text":text_input,"last_para_lang":target_lang})
                st.session_state.paraphrasing_history.append({
                    'timestamp': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    'input': text_input[:100]+("..." if len(text_input)>100 else ""),
                    'paraphrase': para, 'complexity': complexity, 'style': style,
                    'model': model_type, 'lang': target_lang
                })
                save_history_to_db(username,"Paraphrase",model_type,
                    f"Complexity:{complexity}|Style:{style}|Lang:{target_lang}",text_input,para,target_lang)
    if 'last_para' in st.session_state:
        st.markdown("---")
        if st.session_state.get('last_para_lang','English') != 'English':
            st.info(f"🌐  Output in **{st.session_state.last_para_lang}**")
        r1, r2 = st.columns(2)
        with r1:
            st.markdown("**📄 Original**")
            st.markdown(f'<div class="result-box-original">{st.session_state.last_para_text}</div>', unsafe_allow_html=True)
            st.caption(f"📊 {len(st.session_state.last_para_text.split())} words")
        with r2:
            st.markdown("**🔄 Paraphrased**")
            st.markdown(f'<div class="result-box-output">{st.session_state.last_para}</div>', unsafe_allow_html=True)
            st.caption(f"📊 {len(st.session_state.last_para.split())} words")
            st.markdown("### ⭐ Rate this paraphrase")

            rating = st.radio(
               "Your Feedback",
                [1,2,3,4,5],
            horizontal=True,
            key="para_feedback_main"
            )

            if st.button("Submit Feedback", key="para_feedback_btn"):
                 save_feedback(username, "Paraphrase", rating)
                 st.success("Feedback saved!")

        with st.expander("📜  Session History"):
            for item in reversed(st.session_state.paraphrasing_history[-5:]):
                st.markdown(f"**{item['timestamp']}** — `{item['complexity']}` `{item['style']}` `{item['model']}`")
                st.info(item['input']); st.success(item['paraphrase']);
                st.markdown("---")

                st.markdown("### ⭐ Rate this paraphrase")

                rating = st.radio(
                  "Your Feedback",
                  [1,2,3,4,5],
                horizontal=True,
                key="para_fb_radio"
                )

                if st.button("Submit Feedback", key="para_fb"):
                    save_feedback(username, "Paraphrase", rating)
                    st.success("Feedback saved!")

def _simulate_training_metrics(model_arch, epochs, learning_rate, batch_size, dropout_rate, quantization):
    random.seed(hash(f"{model_arch}{epochs}{learning_rate}{batch_size}{dropout_rate}{quantization}"))
    lr_val       = float(learning_rate)
    base_loss    = {"T5-Small":0.55,"BART-Base":0.48,"FLAN-T5":0.42}.get(model_arch, 0.50)
    epoch_factor = 1.0 - (min(epochs,10)*0.06)
    lr_factor    = 1.0 - (lr_val*8000)
    dropout_bonus= dropout_rate*0.08
    quant_penalty= {"FP16 (None)":0.0,"8-bit":0.02,"4-bit":0.05}.get(quantization, 0.0)
    final_loss   = round(max(0.15, base_loss*epoch_factor*lr_factor+dropout_bonus+quant_penalty+random.uniform(-0.03,0.03)),2)
    accuracy     = round(min(95, 65+(epochs*2.5)+(1-final_loss)*20+random.uniform(-2,3)),1)
    rouge_l      = round(random.uniform(1.5,4.0)+epochs*0.15,1)
    bleu         = round(0.25+(epochs*0.02)+(1-final_loss)*0.15+random.uniform(-0.03,0.03),2)
    loss_curve   = []
    curr = base_loss+1.0
    for _ in range(epochs):
        curr = curr*(0.6+random.uniform(-0.05,0.05))+random.uniform(-0.02,0.02)
        loss_curve.append(round(max(final_loss,curr),3))
    loss_curve[-1] = final_loss
    return {"final_loss":str(final_loss),"delta_loss":str(round(random.uniform(-0.08,-0.15),2)),
            "accuracy":f"{accuracy}%","delta_acc":f"+{round(random.uniform(1,6),1)}%",
            "rouge_l":f"+{rouge_l}","delta_rouge":f"+{round(random.uniform(0.3,1.2),1)}",
            "bleu":str(bleu),"delta_bleu":f"+{round(random.uniform(0.02,0.08),2)}",
            "loss_curve":loss_curve,"epochs_x":list(range(1,epochs+1))}

def augmentation_page():
    page_header("🗃️", "Dataset Augmentation & Fine-Tuning", "Build datasets and fine-tune models on custom data")
    tab_explore, tab_tune, tab_studio = st.tabs(["📊  Dataset Explorer","🛠️  Model Tuning","🧪  Augmentation Studio"])
    with tab_explore:
        datasets = {"CNN/DailyMail":{"samples":311029,"type":"News Summarization","avg_words":781},
                    "XSum":         {"samples":226711,"type":"Extreme Summarization","avg_words":431},
                    "PAWS":         {"samples":108461,"type":"Paraphrase","avg_words":21}}
        selected = st.selectbox("📂  Select Dataset", list(datasets.keys()))
        c1,c2,c3 = st.columns(3)
        c1.metric("Total Samples",  f"{datasets[selected]['samples']:,}")
        c2.metric("Task Type",       datasets[selected]['type'])
        c3.metric("Avg Doc Length", f"{datasets[selected]['avg_words']} words")
        st.markdown("<h4 style='color:white'>🧹 Cleaning Filters</h4>", unsafe_allow_html=True)
        col_a, col_b = st.columns(2)
        with col_a: min_len = st.slider("Min Words",5,100,10)
        with col_b: max_len = st.slider("Max Words",100,2000,1000)
        filtered = int(datasets[selected]['samples']*(0.9-(min_len/1000)-(1000-max_len)/2000))
        st.success(f"✅  Filtered size: **{filtered:,} pairs**")
        mock = {"ID":[f"{selected[:3]}-00{i}" for i in range(1,5)],
                "Original Text":[f"Sample text {i} from {selected}." for i in range(1,5)],
                "Target Output":[f"Cleaned target {i}." for i in range(1,5)],
                "Words":[140,432,21,89], "Complexity":[8.4,12.1,4.2,7.9]}
        st.dataframe(pd.DataFrame(mock), use_container_width=True, hide_index=True)
    with tab_tune:
        c1,c2,c3 = st.columns(3)
        with c1:
            model_arch = st.selectbox("🏗️  Architecture", ["T5-Small","BART-Base","FLAN-T5"])
            epochs     = st.slider("🔁  Epochs",1,10,3)
        with c2:
            quantization = st.selectbox("⚡  Quantization", ["FP16 (None)","8-bit","4-bit"])
            batch_size   = st.slider("📦  Batch Size",8,32,16)
        with c3:
            learning_rate = st.selectbox("📉  Learning Rate", ["1e-5","2e-5","3e-5"])
            dropout_rate  = st.slider("💧  Dropout",0.0,0.5,0.1)
        if st.button("🚀  Start Fine-Tuning", type="primary", use_container_width=True):
            with st.spinner(f"Fine-tuning {model_arch}..."):
                bar = st.progress(0)
                for i in range(100): time.sleep(0.01); bar.progress(i+1)
                st.success(f"✅  {model_arch} fine-tuned successfully!")
                metrics = _simulate_training_metrics(model_arch,epochs,learning_rate,batch_size,dropout_rate,quantization)
                m1,m2,m3,m4 = st.columns(4)
                m1.metric("Final Loss",metrics["final_loss"],metrics["delta_loss"])
                m2.metric("Train Acc.",metrics["accuracy"],metrics["delta_acc"])
                m3.metric("ROUGE-L",  metrics["rouge_l"],  metrics["delta_rouge"])
                m4.metric("BLEU",     metrics["bleu"],     metrics["delta_bleu"])
                fig = go.Figure(go.Scatter(x=metrics["epochs_x"],y=metrics["loss_curve"],
                    mode='lines+markers',line=dict(color='#e74c3c',width=3),
                    marker=dict(size=8,color='#e74c3c',line=dict(color='#ff8a8a',width=1.5))))
                fig.update_layout(title=f"Training Loss — {model_arch}",xaxis_title="Epoch",yaxis_title="Loss",
                    paper_bgcolor="rgba(0,0,0,0)",plot_bgcolor="rgba(0,0,0,0)",
                    font=dict(color="#c09090",family="Inter"),height=300,
                    margin=dict(l=0,r=0,t=40,b=0))
                st.plotly_chart(fig, use_container_width=True)
    with tab_studio:
        aug_input = st.text_area("Paragraphs (separated by blank lines):", height=180, key="aug_text",
            value="The quick brown fox jumps over the lazy dog.\n\nAI is transforming the modern world rapidly.")
        col_a, col_b = st.columns(2)
        with col_a: aug_type = st.selectbox("🔧  Transform", ["Paraphrasing","Summarization"])
        with col_b:
            aug_setting = st.selectbox("⚙️  Setting",
                ["Short","Medium","Long"] if aug_type=="Summarization" else ["Advanced","Simple","Neutral"])
        if st.button("🚀  Generate Dataset", type="secondary", use_container_width=True):
            paras = [p.strip() for p in aug_input.split('\n\n') if len(p.strip())>10]
            if not paras: st.error("Enter at least one paragraph.")
            else:
                results = []; bar = st.progress(0)
                for idx, para in enumerate(paras):
                    res = engine.local_summarize(para,aug_setting,"BART",st.session_state.summarization_models) \
                          if aug_type=="Summarization" \
                          else engine.paraphrase_with_model(para,aug_setting,"Creative","FLAN-T5",st.session_state.paraphrase_models)
                    results.append({"#":idx+1,"Original":para[:200],"Generated":res[:200],
                                    "Orig WC":len(para.split()),"Gen WC":len(res.split()),
                                    "Delta":f"{len(res.split())-len(para.split()):+d}"})
                    bar.progress((idx+1)/len(paras))
                st.success(f"✅  Generated {len(results)} pairs!")
                st.dataframe(pd.DataFrame(results), use_container_width=True, hide_index=True)
                st.download_button("📥  Download CSV",
                    data=pd.DataFrame(results).to_csv(index=False).encode('utf-8'),
                    file_name="augmented_dataset.csv", mime="text/csv", use_container_width=True)


# ══════════════════════════════════════════════════════════
# HISTORY PAGE
# ══════════════════════════════════════════════════════════
def history_page(username, is_admin=False):
    page_header("🕘", "Activity History", "Complete log of all your AI operations")

    raw  = get_all_history_from_db(300) if is_admin else get_history_from_db(username, 200)
    if is_admin:
        rows = [{"_username":r[0],"action_type":r[1],"model_used":r[2],"settings":r[3],
                 "input_snippet":r[4],"output_snippet":r[5],"language":r[6],
                 "wc_in":r[7],"wc_out":r[8],"cc_in":r[9],"cc_out":r[10],"created_at":r[11]} for r in raw]
    else:
        rows = [{"_id":r[0],"action_type":r[1],"model_used":r[2],"settings":r[3],
                 "input_snippet":r[4],"output_snippet":r[5],"language":r[6],
                 "wc_in":r[7],"wc_out":r[8],"cc_in":r[9],"cc_out":r[10],"created_at":r[11]} for r in raw]

    # ── Helper functions ──────────────────────────────────
    def reading_time(wc):
        mins = max(1, round(wc / 200))
        return f"{mins} min read"

    def compression_ratio(wc_in, wc_out):
        if not wc_in or wc_in == 0: return "—"
        ratio = round((1 - wc_out / wc_in) * 100, 1)
        return f"{ratio}% ↓" if ratio > 0 else f"{abs(ratio)}% ↑"

    def fmt_dt(dt):
        if not dt: return "N/A"
        try:
            if isinstance(dt, str): dt = datetime.datetime.strptime(dt[:19], "%Y-%m-%d %H:%M:%S")
            # Strip tz if present (DB returns UTC naive or aware), compare with utcnow
            if hasattr(dt, 'tzinfo') and dt.tzinfo is not None:
                dt = dt.replace(tzinfo=None)
            return (dt + datetime.timedelta(hours=5, minutes=30)).strftime("%d %b %Y  %I:%M:%S %p IST")
        except: return str(dt)

    # ── Stat cards ────────────────────────────────────────
    total    = len(rows)
    n_sum    = sum(1 for r in rows if r["action_type"] == "Summarize")
    n_para   = sum(1 for r in rows if r["action_type"] == "Paraphrase")
    n_read   = sum(1 for r in rows if r["action_type"] == "Readability")
    total_wc = sum((r["wc_in"] or 0) for r in rows)
    total_cc = sum((r["cc_in"] or 0) for r in rows)
    top_model = "—"
    if rows:
        from collections import Counter
        mc = Counter(r["model_used"] for r in rows if r["model_used"])
        top_model = mc.most_common(1)[0][0] if mc else "—"

    s1,s2,s3,s4,s5,s6 = st.columns(6)
    for col, num, label, icon in [
        (s1, total,           "Total Actions",    "📋"),
        (s2, n_sum,           "Summaries",        "📝"),
        (s3, n_para,          "Paraphrases",      "🔄"),
        (s4, n_read,          "Readability",      "📊"),
        (s5, f"{total_wc:,}", "Words Processed",  "📖"),
        (s6, top_model,       "Top Model",        "🤖"),
    ]:
        col.markdown(f"""<div class="stat-card">
            <div style="font-size:22px;margin-bottom:4px;">{icon}</div>
            <div class="stat-num">{num}</div>
            <div class="stat-label">{label}</div>
        </div>""", unsafe_allow_html=True)

    st.markdown("<br>", unsafe_allow_html=True)

    # ── Tabs: Table | Cards | Analytics ──────────────────
    tab_table, tab_cards, tab_analytics = st.tabs(["📋  Table View", "🗂️  Card View", "📈  Analytics"])

    # ── Shared filter bar ────────────────────────────────
    with tab_table:
        _render_filters_and_table(rows, username, is_admin, fmt_dt, reading_time, compression_ratio)

    with tab_cards:
        _render_card_view(rows, username, is_admin, fmt_dt, reading_time, compression_ratio)

    with tab_analytics:
        _render_analytics(rows)


def _render_filters_and_table(rows, username, is_admin, fmt_dt, reading_time, compression_ratio):
    cf1,cf2,cf3,cf4 = st.columns([2,2,2,1])
    with cf1: search_q    = st.text_input("🔍 Search", placeholder="keyword...", key="tbl_search")
    with cf2: filter_type = st.selectbox("Type", ["All","Summarize","Paraphrase","Readability"], key="tbl_type")
    with cf3: filter_model= st.selectbox("Model",
                ["All"]+sorted({r["model_used"] for r in rows if r["model_used"]}), key="tbl_model")
    with cf4: per_page    = st.selectbox("Per page",[10,25,50], key="tbl_pp")

    if not is_admin:
        if st.button("🗑️  Clear My History", type="secondary", key="clear_tbl"):
            clear_user_history(username); st.success("Cleared."); time.sleep(0.4); st.rerun()

    filtered = _apply_filters(rows, filter_type, filter_model, search_q)
    if not filtered:
        st.info("📭  No records match your filters."); return

    # ── Pagination ──────────────────────────────────────
    total_pages = max(1, (len(filtered) + per_page - 1) // per_page)
    if "tbl_page" not in st.session_state: st.session_state["tbl_page"] = 1
    st.session_state["tbl_page"] = min(st.session_state["tbl_page"], total_pages)
    page_num  = st.session_state["tbl_page"]
    start_i   = (page_num - 1) * per_page
    page_rows = filtered[start_i : start_i + per_page]

    st.markdown(
        f'<div style="color:white;font-size:12px;margin-bottom:8px;">'
        f'Showing {start_i+1}–{min(start_i+per_page,len(filtered))} of {len(filtered)} '
        f'&nbsp;·&nbsp; Page {page_num}/{total_pages} '
        f'&nbsp;·&nbsp; <span style="color:#c08080;">Click any row to expand full details</span></div>',
        unsafe_allow_html=True)

    # ── Build compact DataFrame (short previews, small cols) ──
    icon_map = {"Summarize":"📝","Paraphrase":"🔄","Readability":"📊"}
    table_rows = []
    for idx_r, r in enumerate(page_rows):
        wc_in  = r.get("wc_in",0)  or 0
        wc_out = r.get("wc_out",0) or 0
        row = {
            "#":              start_i + idx_r + 1,
            "Date":           fmt_dt(r.get("created_at")),
            "Type":           icon_map.get(r["action_type"],"") + " " + r["action_type"],
            "Model":          (r.get("model_used") or "—"),
            "Lang":           (r.get("language") or "EN"),
            "W.In":           wc_in,
            "W.Out":          wc_out,
            "Compress":       compression_ratio(wc_in, wc_out),
            "Input ↓":        (r.get("input_snippet")  or "")[:45] + ("…" if len(r.get("input_snippet") or "") > 45 else ""),
            "Output ↓":       (r.get("output_snippet") or "")[:45] + ("…" if len(r.get("output_snippet") or "") > 45 else ""),
        }
        if is_admin and "_username" in r: row["User"] = r["_username"]
        table_rows.append(row)

    df = pd.DataFrame(table_rows)

    def row_style(row):
        t = str(row.get("Type",""))
        if "Summarize"    in t: bg = "background-color:rgba(192,57,43,0.08);"
        elif "Paraphrase" in t: bg = "background-color:rgba(180,80,30,0.08);"
        elif "Readability"in t: bg = "background-color:rgba(140,30,30,0.10);"
        else: bg = ""
        return [bg] * len(row)

    col_cfg = {
        "#":         st.column_config.NumberColumn("#",       width=40),
        "Date":      st.column_config.TextColumn("🕐 Date",   width=160),
        "Type":      st.column_config.TextColumn("Type",      width=120),
        "Model":     st.column_config.TextColumn("Model",     width=90),
        "Lang":      st.column_config.TextColumn("Lang",      width=70),
        "W.In":      st.column_config.NumberColumn("W.In",    width=55, format="%d"),
        "W.Out":     st.column_config.NumberColumn("W.Out",   width=60, format="%d"),
        "Compress":  st.column_config.TextColumn("Compress",  width=80),
        "Input ↓":   st.column_config.TextColumn("Input ↓",  width=200),
        "Output ↓":  st.column_config.TextColumn("Output ↓", width=200),
    }
    if is_admin: col_cfg["User"] = st.column_config.TextColumn("User", width=90)

    event = st.dataframe(
        df.style.apply(row_style, axis=1),
        use_container_width=True,
        hide_index=True,
        column_config=col_cfg,
        height=400,
        on_select="rerun",
        selection_mode="single-row",
        key="hist_table_sel",
    )

    # ── Expand selected row ─────────────────────────────
    selected_rows = event.selection.rows if hasattr(event, "selection") else []
    if selected_rows:
        sel_idx = selected_rows[0]
        r = page_rows[sel_idx]
        wc_in  = r.get("wc_in",0)  or 0
        wc_out = r.get("wc_out",0) or 0
        cc_in  = r.get("cc_in",0)  or 0
        cc_out = r.get("cc_out",0) or 0
        atype  = r["action_type"]
        badge_map = {"Summarize":("#e74c3c","📝"),"Paraphrase":("#c0392b","🔄"),"Readability":("#a93226","📊")}
        bclr, bicon = badge_map.get(atype,("#7b241c","📋"))

        st.markdown(f"""
        <div style="background:rgba(192,57,43,0.07);border:1px solid rgba(231,76,60,0.35);
                    border-radius:14px;padding:22px 26px;margin-top:12px;">
          <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:16px;flex-wrap:wrap;gap:10px;">
            <div style="display:flex;align-items:center;gap:12px;">
              <span style="background:{bclr};color:#fff;padding:5px 14px;border-radius:20px;
                           font-size:13px;font-weight:600;">{bicon} {atype}</span>
              <span style="color:#ff6b6b;font-size:15px;font-weight:600;">Row #{start_i+sel_idx+1} — Full Detail</span>
            </div>
            <span style="color:#7a5050;font-size:12px;">🕐 {fmt_dt(r.get("created_at"))}</span>
          </div>
          <div style="display:flex;gap:24px;flex-wrap:wrap;margin-bottom:14px;">
            <span style="color:#e08060;font-size:13px;">🤖 {r.get("model_used") or "—"}</span>
            <span style="color:#d07070;font-size:13px;">🌐 {r.get("language") or "English"}</span>
            <span style="color:#9a6060;font-size:13px;">⚙️ {r.get("settings") or "—"}</span>
            <span style="color:#8a5050;font-size:13px;">📥 {wc_in} words / {cc_in:,} chars</span>
            <span style="color:#a06060;font-size:13px;">📤 {wc_out} words / {cc_out:,} chars</span>
            <span style="color:#c07070;font-size:13px;">📊 Compression: {compression_ratio(wc_in, wc_out)}</span>
            <span style="color:#b07070;font-size:13px;">⏱️ {reading_time(wc_out)}</span>
          </div>
        </div>""", unsafe_allow_html=True)

        ec1, ec2 = st.columns(2)
        with ec1:
            st.markdown("**📄 Full Input**")
            st.markdown(
                f'<div class="result-box-original" style="max-height:260px;overflow-y:auto;'
                f'font-size:13px;line-height:1.6;">{r.get("input_snippet") or "<em>No input stored</em>"}</div>',
                unsafe_allow_html=True)
        with ec2:
            st.markdown("**✨ Full Output**")
            st.markdown(
                f'<div class="result-box-output" style="max-height:260px;overflow-y:auto;'
                f'font-size:13px;line-height:1.6;">{r.get("output_snippet") or "<em>No output stored</em>"}</div>',
                unsafe_allow_html=True)
    else:
        st.markdown(
            '<div style="text-align:center;color:#5a3030;font-size:12px;padding:8px;'
            'border:1px dashed rgba(192,57,43,0.2);border-radius:8px;margin-top:8px;">'
            '👆 Click any row above to see full input & output details</div>',
            unsafe_allow_html=True)

    # ── Pagination controls ─────────────────────────────
    st.markdown("<br>", unsafe_allow_html=True)
    pc1,pc2,pc3,pc4,pc5 = st.columns([1,1,2,1,1])
    with pc1:
        if st.button("⏮️", use_container_width=True, key="pg_first", help="First page"):
            st.session_state["tbl_page"]=1; st.rerun()
    with pc2:
        if st.button("◀", use_container_width=True, key="pg_prev", help="Previous"):
            st.session_state["tbl_page"]=max(1,page_num-1); st.rerun()
    with pc3:
        st.markdown(
            f'<div style="text-align:center;padding:8px;color:#c08080;font-size:13px;">'
            f'Page <b style="color:#ff6b6b">{page_num}</b> / <b style="color:#ff6b6b">{total_pages}</b></div>',
            unsafe_allow_html=True)
    with pc4:
        if st.button("▶", use_container_width=True, key="pg_next", help="Next"):
            st.session_state["tbl_page"]=min(total_pages,page_num+1); st.rerun()
    with pc5:
        if st.button("⏭️", use_container_width=True, key="pg_last", help="Last page"):
            st.session_state["tbl_page"]=total_pages; st.rerun()

    # ── Export ──────────────────────────────────────────
    ist_now = (datetime.datetime.utcnow()+datetime.timedelta(hours=5,minutes=30)).strftime("%Y%m%d_%H%M")
    full_df = _build_full_df(filtered, is_admin, fmt_dt, reading_time, compression_ratio, icon_map)
    st.download_button("📥  Export All as CSV",
        data=full_df.to_csv(index=False).encode("utf-8"),
        file_name=f"history_{username}_{ist_now}.csv",
        mime="text/csv", use_container_width=True)

def _render_card_view(rows, username, is_admin, fmt_dt, reading_time, compression_ratio):
    cf1,cf2,cf3 = st.columns([2,2,2])
    with cf1: search_q    = st.text_input("🔍 Search", placeholder="keyword...", key="card_search")
    with cf2: filter_type = st.selectbox("Type",["All","Summarize","Paraphrase","Readability"],key="card_type")
    with cf3: filter_model= st.selectbox("Model",
                ["All"]+sorted({r["model_used"] for r in rows if r["model_used"]}),key="card_model")

    if not is_admin:
        if st.button("🗑️  Clear My History", type="secondary", key="clear_card"):
            clear_user_history(username); st.success("Cleared."); time.sleep(0.4); st.rerun()

    filtered = _apply_filters(rows, filter_type, filter_model, search_q)
    if not filtered:
        st.markdown("""<div style="text-align:center;padding:48px;color:#5a3030;">
            <div style="font-size:44px;">📭</div>
            <div style="margin-top:10px;font-size:15px;">No records found</div>
        </div>""", unsafe_allow_html=True); return

    st.markdown(f'<div style="color:#7a5050;font-size:13px;margin-bottom:12px;">{len(filtered)} records</div>',
                unsafe_allow_html=True)

    badge_map = {"Summarize":  ("hist-badge-summarize",  "📝 Summarize"),
                 "Paraphrase": ("hist-badge-paraphrase", "🔄 Paraphrase"),
                 "Readability":("hist-badge-readability","📊 Readability")}

    for r in filtered[:50]:   # show max 50 in card view
        atype  = r["action_type"]
        bclass, blabel = badge_map.get(atype, ("hist-badge-summarize", atype))
        model  = r.get("model_used") or "N/A"
        lang   = r.get("language")   or "English"
        sett   = r.get("settings")   or ""
        inp    = r.get("input_snippet")  or ""
        out    = r.get("output_snippet") or ""
        wc_in  = r.get("wc_in", 0)  or 0
        wc_out = r.get("wc_out", 0) or 0
        cc_in  = r.get("cc_in", 0)  or 0
        cc_out = r.get("cc_out", 0) or 0
        ts     = fmt_dt(r.get("created_at"))
        cr     = compression_ratio(wc_in, wc_out)
        rt     = reading_time(wc_out)
        user_lbl = f" &nbsp;·&nbsp; 👤 {r['_username']}" if is_admin and "_username" in r else ""
        uid    = r.get("_id", id(r))

        with st.container():
            st.markdown(f"""<div class="hist-card">
              <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:8px;">
                <span class="{bclass}">{blabel}</span>
                <div style="display:flex;gap:12px;align-items:center;flex-wrap:wrap;">
                  <span style="background:rgba(255,255,255,0.12);color:rgba(255,255,255,0.85);padding:3px 10px;border-radius:16px;font-size:11px;border:1px solid rgba(192,57,43,0.25);">📊 {cr}</span>
                  <span style="background:rgba(255,255,255,0.12);color:#c09080;padding:3px 10px;border-radius:16px;font-size:11px;border:1px solid rgba(192,57,43,0.2);">⏱️ {rt}</span>
                  <span class="hist-ts">🕐 {ts}{user_lbl}</span>
                </div>
              </div>
              <div style="margin-top:10px;display:flex;gap:18px;flex-wrap:wrap;align-items:center;">
                <span class="hist-model">🤖 {model}</span>
                <span class="hist-lang">🌐 {lang}</span>
                <span style="color:#6a4040;font-size:12px;">⚙️ {sett}</span>
              </div>
              <div style="margin-top:8px;display:flex;gap:16px;flex-wrap:wrap;">
                <span style="color:rgba(255,255,255,0.85);font-size:12px;">📥 {wc_in} words / {cc_in:,} chars</span>
                <span style="color:rgba(255,255,255,0.85);font-size:12px;">📤 {wc_out} words / {cc_out:,} chars</span>
              </div>
              <div class="hist-snip"><span style="color:#6a4040;font-weight:600;">Input:</span> {inp[:220] or "<em style='color:#4a3030'>—</em>"}</div>
              <div class="hist-out"><span style="color:#b06060;font-weight:600;">Output:</span> {out[:220] or "<em style='color:#4a3030'>—</em>"}</div>
            </div>""", unsafe_allow_html=True)

            with st.expander(f"🔍  Full Text — {ts}"):
                ec1, ec2 = st.columns(2)
                with ec1:
                    st.markdown("**📄 Full Input**")
                    st.markdown(f'<div class="result-box-original" style="max-height:300px;overflow-y:auto;">{inp}</div>', unsafe_allow_html=True)
                with ec2:
                    st.markdown("**✨ Full Output**")
                    st.markdown(f'<div class="result-box-output" style="max-height:300px;overflow-y:auto;">{out}</div>', unsafe_allow_html=True)

    if len(filtered) > 50:
        st.info(f"Showing first 50 of {len(filtered)} cards. Use Table View for full list.")


def _render_analytics(rows):
    if not rows:
        st.info("📭  No data yet to analyze."); return

    from collections import Counter
    import plotly.express as px

    # ── Row 1: Action type breakdown + Model usage ────
    a1, a2 = st.columns(2)

    with a1:
        st.markdown("#### Action Type Distribution")
        type_counts = Counter(r["action_type"] for r in rows)
        fig_pie = go.Figure(go.Pie(
            labels=list(type_counts.keys()),
            values=list(type_counts.values()),
            marker=dict(colors=["#e74c3c","#c0392b","#a93226"],
                        line=dict(color="#1a0808",width=2)),
            textfont=dict(color="#f0d0d0",size=13),
            hole=0.45,
        ))
        fig_pie.update_layout(paper_bgcolor="#120808",font=dict(color="#c09090",family="Inter"),
                              height=280,margin=dict(l=10,r=10,t=30,b=10),
                              legend=dict(font=dict(color="#c09090"),bgcolor="rgba(0,0,0,0)"))
        st.plotly_chart(fig_pie, use_container_width=True)

    with a2:
        st.markdown("#### Model Usage")
        model_counts = Counter(r["model_used"] for r in rows if r["model_used"])
        fig_bar = go.Figure(go.Bar(
            x=list(model_counts.values()), y=list(model_counts.keys()),
            orientation="h",
            marker=dict(color=["#e74c3c","#c0392b","#a93226","#922b21","#7b241c"][:len(model_counts)],
                        line=dict(color="#1a0808",width=1)),
            text=list(model_counts.values()), textposition="auto",
            textfont=dict(color="#fff",size=12)
        ))
        fig_bar.update_layout(paper_bgcolor="#120808",plot_bgcolor="#1a0a0a",
                              font=dict(color="#c09090",family="Inter"),
                              xaxis=dict(gridcolor="#2a1010"),yaxis=dict(gridcolor="#2a1010"),
                              height=280,margin=dict(l=10,r=10,t=30,b=10))
        st.plotly_chart(fig_bar, use_container_width=True)

    # ── Row 2: Words processed over time ──────────────
    st.markdown("#### Words Processed Over Time")
    time_data = []
    for r in rows:
        dt = r.get("created_at")
        if dt:
            try:
                if isinstance(dt, str): dt = datetime.datetime.strptime(dt[:19],"%Y-%m-%d %H:%M:%S")
                if hasattr(dt,"tzinfo") and dt.tzinfo: dt = dt.replace(tzinfo=None)
                ist = dt + datetime.timedelta(hours=5,minutes=30)
                time_data.append({"date": ist.strftime("%d %b"), "wc": r.get("wc_in",0) or 0,
                                   "type": r["action_type"]})
            except: pass

    if time_data:
        df_time = pd.DataFrame(time_data)
        df_grouped = df_time.groupby(["date","type"])["wc"].sum().reset_index()
        colors = {"Summarize":"#e74c3c","Paraphrase":"#c0392b","Readability":"#7b241c"}
        fig_line = go.Figure()
        for atype, clr in colors.items():
            d = df_grouped[df_grouped["type"]==atype]
            if not d.empty:
                fig_line.add_trace(go.Scatter(x=d["date"],y=d["wc"],mode="lines+markers",
                    name=atype,line=dict(color=clr,width=2.5),
                    marker=dict(size=7,color=clr,line=dict(color="#ff8a8a",width=1))))
        fig_line.update_layout(paper_bgcolor="#120808",plot_bgcolor="#1a0a0a",
                               font=dict(color="#c09090",family="Inter"),
                               xaxis=dict(gridcolor="#2a1010"),yaxis=dict(gridcolor="#2a1010",title="Words"),
                               legend=dict(bgcolor="rgba(0,0,0,0)",font=dict(color="#c09090")),
                               height=300,margin=dict(l=10,r=10,t=10,b=10))
        st.plotly_chart(fig_line, use_container_width=True)

    # ── Row 3: Avg compression + Language breakdown ───
    b1, b2 = st.columns(2)
    with b1:
        st.markdown("#### Avg Compression by Model")
        model_comp = {}
        for r in rows:
            m = r.get("model_used") or "Other"
            wi, wo = (r.get("wc_in",0) or 0), (r.get("wc_out",0) or 0)
            if wi > 0:
                if m not in model_comp: model_comp[m] = []
                model_comp[m].append((1 - wo/wi)*100)
        if model_comp:
            mc_avg = {m: round(sum(v)/len(v),1) for m,v in model_comp.items()}
            fig_comp = go.Figure(go.Bar(
                x=list(mc_avg.keys()), y=list(mc_avg.values()),
                marker_color=["#e74c3c","#c0392b","#a93226","#922b21","#7b241c"][:len(mc_avg)],
                text=[f"{v}%" for v in mc_avg.values()], textposition="outside",
                textfont=dict(color="#d0a0a0",size=12)
            ))
            fig_comp.update_layout(paper_bgcolor="#120808",plot_bgcolor="#1a0a0a",
                                   font=dict(color="#c09090",family="Inter"),
                                   yaxis=dict(title="Avg Compression %",gridcolor="#2a1010"),
                                   xaxis=dict(gridcolor="#2a1010"),
                                   height=280,margin=dict(l=10,r=10,t=20,b=10))
            st.plotly_chart(fig_comp, use_container_width=True)
        else:
            st.info("Need summarize/paraphrase data for compression stats.")

    with b2:
        st.markdown("#### Language Distribution")
        lang_counts = Counter(r.get("language","English") for r in rows)
        fig_lang = go.Figure(go.Pie(
            labels=list(lang_counts.keys()),
            values=list(lang_counts.values()),
            marker=dict(colors=["#e74c3c","#c0392b","#a93226","#922b21","#7b241c","#641e16","#4a0f0f"][:len(lang_counts)],
                        line=dict(color="#1a0808",width=2)),
            textfont=dict(color="#f0d0d0",size=12), hole=0.4,
        ))
        fig_lang.update_layout(paper_bgcolor="#120808",font=dict(color="#c09090",family="Inter"),
                               height=280,margin=dict(l=10,r=10,t=10,b=10),
                               legend=dict(font=dict(color="#c09090"),bgcolor="rgba(0,0,0,0)"))
        st.plotly_chart(fig_lang, use_container_width=True)


def _apply_filters(rows, filter_type, filter_model, search_q):
    filtered = rows
    if filter_type  != "All": filtered = [r for r in filtered if r["action_type"]==filter_type]
    if filter_model != "All": filtered = [r for r in filtered if r.get("model_used")==filter_model]
    if search_q:
        q = search_q.lower()
        filtered = [r for r in filtered if q in (r.get("input_snippet") or "").lower()
                    or q in (r.get("output_snippet") or "").lower()]
    return filtered


def _build_full_df(filtered, is_admin, fmt_dt, reading_time, compression_ratio, icon_map):
    table_rows = []
    for r in filtered:
        wc_in  = r.get("wc_in",0)  or 0
        wc_out = r.get("wc_out",0) or 0
        cc_in  = r.get("cc_in",0)  or 0
        cc_out = r.get("cc_out",0) or 0
        row = {
            "Date & Time (IST)": fmt_dt(r.get("created_at")),
            "Type":              icon_map.get(r["action_type"],"") + " " + r["action_type"],
            "Model":             r.get("model_used") or "—",
            "Settings":          r.get("settings")   or "—",
            "Language":          r.get("language")   or "English",
            "Words In":          wc_in,
            "Words Out":         wc_out,
            "Compression":       compression_ratio(wc_in, wc_out),
            "Read Time":         reading_time(wc_out),
            "Chars In":          cc_in,
            "Chars Out":         cc_out,
            "Input":             r.get("input_snippet")  or "",
            "Output":            r.get("output_snippet") or "",
        }
        if is_admin and "_username" in r: row["User"] = r["_username"]
        table_rows.append(row)
    return pd.DataFrame(table_rows)

#==================================
# Profile_Page
#==================================

def profile_page():

    token = st.session_state["jwt_token"]
    payload = verify_token(token)

    username = payload.get("username")
    current_email = payload.get("sub")

    page_header("👤", "User Profile")

    db_email, avatar = get_user_profile(username)
    col1, col2 = st.columns([1,2])

    with col1:

        if avatar:
            st.image(base64.b64decode(avatar), width=150)
        else:
            st.info("No profile picture")

        uploaded_avatar = st.file_uploader(
            "Upload Profile Picture",
            type=["png","jpg","jpeg"],
            key="profile_pic"
        )

        if uploaded_avatar:
            st.image(uploaded_avatar, width=120)
            if st.button("Update Picture"):
                  avatar_bytes = uploaded_avatar.getvalue()
                  avatar_base64 = base64.b64encode(avatar_bytes).decode()

                  update_user_avatar(username, avatar_base64)

                  st.success("Profile picture updated!")
                  st.rerun()



    with col2:

        st.markdown(f"### {username}")

        st.markdown(f"📧 **Current Email:** {current_email}")

        st.markdown("---")

        # -----------------------
        # Update Email
        # -----------------------

        st.markdown("### ✉️ Update Email")

        new_email = st.text_input("New Email")

        if st.button("Update Email"):

              if new_email:

                  ok = update_user_email(username, new_email)

                  if ok:

                       # update email inside JWT token
                       payload["sub"] = new_email
                       st.session_state["jwt_token"] = create_access_token(payload)

                       st.success("Email updated successfully")
                       st.rerun()



        st.markdown("---")

        # -----------------------
        # Change Password
        # -----------------------

        st.markdown("### 🔑 Change Password")

        new_pass = st.text_input("New Password", type="password")

        confirm_pass = st.text_input("Confirm Password", type="password")

        if st.button("Update Password"):

            if new_pass != confirm_pass:

                st.error("Passwords do not match")

            else:

                update_password(current_email, new_pass)

                st.success("Password updated successfully")


# ══════════════════════════════════════════════════════════
# ADMIN PAGE
# ══════════════════════════════════════════════════════════
def admin_page():
    payload = verify_token(st.session_state["jwt_token"])
    if not payload: st.session_state.update({"jwt_token":None,"page":"login"}); st.rerun(); return
    if payload.get("username","").lower() != "admin": st.error("⛔ Access Denied"); return
    page_header("🛡️", "Admin Dashboard", "Manage registered users")
    users = get_all_users()
    c1,c2,c3 = st.columns(3)
    c1.metric("👥 Total Users",    len(users))
    c2.metric("🔴 Active Sessions","—")
    c3.metric("📅 Today Joins",    sum(1 for _,_,d in users if d and (datetime.datetime.utcnow()-d).days<1))
    st.markdown("---")
    st.markdown("""<div style="display:grid;grid-template-columns:2fr 3fr 2fr 1fr;gap:8px;
        padding:10px 16px;background:rgba(192,57,43,0.1);border-radius:8px;margin-bottom:8px;
        font-size:13px;font-weight:600;color:#c08080;">
        <span>👤 Username</span><span>📧 Email</span><span>🕐 Joined</span><span>🗑️</span>
    </div>""", unsafe_allow_html=True)
    for uname, uemail, ucreated in users:
        c1, c2, c3, c4 = st.columns([2,3,2,1])
        c1.markdown(f"**{uname}**")
        c2.markdown(f'<span style="color:#c09090;font-size:13px;">{uemail}</span>', unsafe_allow_html=True)
        c3.markdown(f'<span style="color:#8a6060;font-size:13px;">{get_relative_time(ucreated) if ucreated else "N/A"}</span>', unsafe_allow_html=True)
        if uname.lower() != "admin":
            if c4.button("🗑️", key=f"del_{uemail}", help=f"Delete {uname}"):
                delete_user(uemail); st.warning(f"Deleted: {uemail}"); time.sleep(0.4); st.rerun()

# ══════════════════════════════════════════════════════════
# DASHBOARD SHELL
# ══════════════════════════════════════════════════════════
def dashboard_page():

    token = st.session_state["jwt_token"]
    payload = verify_token(token)

    if not payload:
        st.session_state.update({"jwt_token": None, "page": "login"})
        st.rerun()
        return

    username = payload.get("username", "User")
    email = payload.get("sub", "")
    is_admin = username.lower() == "admin"

    if "nav_selected" not in st.session_state:
        st.session_state["nav_selected"] = "Readability"

    with st.sidebar:

        import base64

        email, avatar = get_user_profile(username)

        # ----- Title -----
        st.markdown(
    "<h3 style='margin-left:5px;'>🤖 Infosys LLM</h3>",
    unsafe_allow_html=True
)
        # ----- Avatar -----
        if avatar:
            st.markdown(f"""
            <div style="width:90px;height:90px;border-radius:50%;overflow:hidden;margin:auto">
                <img src="data:image/png;base64,{avatar}" style="width:100%;height:100%;object-fit:cover;">
            </div>
            """, unsafe_allow_html=True)

        else:
            initials = username[0].upper() if username else "U"

            st.markdown(f"""
            <div style="
                width:90px;height:90px;border-radius:50%;
                background:#e74c3c;color:white;
                display:flex;align-items:center;justify-content:center;
                font-size:32px;font-weight:bold;margin:auto">
                {initials}
            </div>
            """, unsafe_allow_html=True)

        st.markdown(f"<div style='text-align:center';margin-bottom:15px;>👤 {username}</div>", unsafe_allow_html=True)

        # ---------- Navigation ----------
        nav_items = [
            ("📊  Readability", "Readability"),
            ("📝  Summarize", "Summarize"),
            ("🔄  Paraphrase", "Paraphrase"),
            ("🗃️  Augment", "Augment"),
            ("🕘  History", "History"),
            ("👤  Profile", "Profile"),
        ]

        if is_admin:
            nav_items.append(("🛡️  Admin", "Admin"))

        for label, key in nav_items:

            btn_type = "primary" if st.session_state["nav_selected"] == key else "secondary"

            if st.button(label, use_container_width=True, type=btn_type, key=f"nav_{key}"):

                st.session_state["nav_selected"] = key
                st.rerun()

        st.markdown("---")

        if st.button("🔓 Sign Out", use_container_width=True):

            st.session_state.update({
                "jwt_token": None,
                "username": None,
                "nav_selected": "Readability",
                "page": "login"
            })

            st.rerun()


    # ---------- Page Routing ----------
    sel = st.session_state["nav_selected"]

    if sel == "Readability":
        readability_page()

    elif sel == "Summarize":
        summarizer_page()

    elif sel == "Paraphrase":
        paraphraser_page()

    elif sel == "Augment":
        augmentation_page()

    elif sel == "History":
        history_page(username, is_admin=is_admin)

    elif sel == "Admin":
        admin_page()

    elif sel == "Profile":
        profile_page()

# ══════════════════════════════════════════════════════════
# ROUTER
# ══════════════════════════════════════════════════════════
if st.session_state.get("jwt_token"):

    payload = verify_token(st.session_state["jwt_token"])

    if payload:

        role = payload.get("role", "user")

        if role == "admin":
            admin_dashboard()
        else:
           dashboard_page()

    else:
        st.session_state.update({
            "jwt_token": None,
            "page": "login"
        })
        st.rerun()

else:

    page = st.session_state.get("page", "login")

    if page == "signup":
        signup_page()

    elif page == "forgot":
        forgot_password_page()

    else:
        login_page()


Writing app.py


In [ ]:
!pkill -f streamlit
!pkill -f ngrok

In [ ]:
# ── Save PostgreSQL database to Google Drive ─────────────────
# Run this cell anytime to backup your DB to Drive
import subprocess, os
BACKUP_PATH = '/content/drive/MyDrive/postgres_milestone_db/milestone1_users.sql'
os.makedirs(os.path.dirname(BACKUP_PATH), exist_ok=True)
result = subprocess.run(
    f'sudo -u postgres pg_dump milestone1_users > "{BACKUP_PATH}"',
    shell=True, capture_output=True, text=True
)
if result.returncode == 0:
    size = os.path.getsize(BACKUP_PATH)
    print(f'✅ Database saved to Google Drive! ({size} bytes)')
    print(f'📁 Path: {BACKUP_PATH}')
else:
    print(f'❌ Backup failed: {result.stderr}')


✅ Database saved to Google Drive! (12519 bytes)
📁 Path: /content/drive/MyDrive/postgres_milestone_db/milestone1_users.sql


In [ ]:
import os, subprocess, time
from pyngrok import ngrok
from google.colab import userdata

ngrok.kill()
time.sleep(5) # Increased sleep duration to allow ngrok server to release previous tunnel

os.environ['EMAIL_PASSWORD'] = userdata.get('EMAIL_PASSWORD')
os.environ['EMAIL_ADDRESS']  = userdata.get('EMAIL_ADDRESS')
os.environ['ADMIN_EMAIL_ID'] = userdata.get('ADMIN_EMAIL_ID')
os.environ['ADMIN_PASSWORD'] = userdata.get('ADMIN_PASSWORD')
os.environ['JWT_secret_key']     = 'super-secret-change-me'
os.environ['DB_NAME']        = 'milestone1_users'
os.environ['DB_USER']        = 'postgres'
os.environ['DB_PASSWORD']    = 'postgres'
os.environ['DB_HOST']        = 'localhost'
os.environ['DB_PORT']        = '5432'

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))

st_proc = subprocess.Popen(
    ['python', '-m', 'streamlit', 'run', 'app.py', '--server.port', '8502', '--server.headless', 'true'],
    env=os.environ.copy()
)
time.sleep(5)

public_url = ngrok.connect(8502).public_url
print(f"🚀 App running at: {public_url}")

input("Press ENTER to stop...")
st_proc.terminate(); ngrok.kill()

🚀 App running at: https://kenda-burliest-vividly.ngrok-free.dev
Press ENTER to stop...
